# Day 13 — RAG operations: tracing, latency, and cost

*Part of **15 Days of Modern, Enterprise-Relevant RAG from Scratch**. Lesson: [`docs/day-13/LEARN.md`](../docs/day-13/LEARN.md).*

Day 12 graded the pipeline's answers. Today asks what each answer **costs** and what happens **when something fails**, and keeps every answer the same while finding out:

1. **A trace for every request**: each step's timing, the ids and scores it produced and what it decided, with no document text, no question text and no key in it.
2. **Latency per step** (p50 and p95): where a request's time goes, cold and warm, and what Day 12's live model added.
3. **Cost**: where a model call's tokens go, checked against the 40 token counts Day 12's live pass recorded.
4. **Caching**: exact caches that can't change an answer, two cache keys that look reasonable and aren't, and a **semantic cache** that can.
5. **Timeouts, retries, a circuit breaker and a fallback** for the optional model client, each failure shown against a local fake server.

Two things today can change an answer, and each is a decision measured on **51 new held-out questions** (`questions-v5`): **D1**, a semantic answer cache, and **D2**, answering with the API-free reader when the model fails. **Every rule was committed before anything scored a held-out question**: [`evals/preregistration/day-13.md`](../evals/preregistration/day-13.md), the first commit on today's branch together with the new questions. The notebook recomputes the dev values and stops if they differ.

Everything runs locally: no API key, no new packages, no new model (Day 04's), about a minute (a few seconds of it are the fake servers' deliberate delays). An optional live pass sends 20 questions through the new client to a hosted model (section 9, off by default).

## 0. Setup

No new packages. The shared code in [`src/rag_from_scratch/`](../src/rag_from_scratch/) gains **`rubric.py`**, Day 12's answer rubric (`grade()`, `value()`, `closest_call()`, `compare_answers()`), because this notebook grades with it; `tests/test_shared_code.py` checks it against the Day 12 notebook. Everything new today is written out here first.

The client's settings are in the pre-registration: three attempts, 10 s per attempt, a 30 s deadline for the whole model step, backoff with jitter between 0.5 s and 8 s, a breaker that opens after three failed requests for 30 s. The local fake servers below use the same logic 20 times faster, so the notebook runs in seconds.

In [1]:
import dataclasses
import hashlib
import json
import math
import os
import platform
import random
import re
import statistics
import sys
import threading
import time
import urllib.error
import urllib.parse
import urllib.request
import warnings
from collections import Counter
from contextlib import contextmanager, nullcontext
from dataclasses import dataclass
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
from pathlib import Path

assert sys.version_info >= (3, 11), "This series needs Python 3.11 or newer."
# huggingface_hub's progress bars warn when Jupyter widgets aren't installed; they still work as plain text.
warnings.filterwarnings("ignore", message="IProgress not found")

try:
    import numpy as np
    import onnxruntime as ort
    import tokenizers
except ImportError as error:
    raise ImportError("Day 13 uses Day 04's packages (numpy, onnxruntime, tokenizers, huggingface-hub). From the "
                      "repository root run: python -m pip install -r requirements.txt") from error


def find_repo_root() -> Path:
    # Walk up until the shared package is found, so the notebook works from the repo root or
    # from notebooks/. No hard-coded paths.
    here = Path.cwd().resolve()
    for folder in (here, *here.parents):
        if (folder / "src" / "rag_from_scratch").is_dir():
            return folder
    raise FileNotFoundError("Could not find src/rag_from_scratch. Start Jupyter from inside the cloned repository.")


REPO_ROOT = find_repo_root()
if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))           # import the shared code without installing it

from rag_from_scratch.answering import (ABSTAIN, BLOCKED, DIRECTORY, MAX_CONTEXT_WORDS, TOP_K, ExtractiveReader,  # Day 06
                                        FixedReply, GeneratorError, RagAnswer, Reply, Source, answer_units, assemble,
                                        check_citations, count_words)
from rag_from_scratch.bm25 import BM25Index                              # Day 01's BM25
from rag_from_scratch.chat import (INSTRUCTIONS, LOCAL_HOSTS, ChatModel, build_messages,  # Day 06's prompt and client
                                   read_env_file)
from rag_from_scratch.chunking import section_chunks                     # Day 03's chunker
from rag_from_scratch.contexts import SentenceCache, answer_context, may_read   # Day 11's end-to-end steps
from rag_from_scratch.corpus import corpus_after                         # Day 05: a changed copy of the documents
from rag_from_scratch.embeddings import QUERY_INSTRUCTION, Embedder      # Day 04's model
from rag_from_scratch.evaluation import (TIE, fingerprint, first_relevant, load_queries, outcome, sign_test,  # Day 07
                                         snapshot, validate, wilson)
from rag_from_scratch.fusion import ALPHA, weighted                      # Day 08's fusion
from rag_from_scratch.index import Result, VectorIndex, groups_filter, sync   # Day 05's index
from rag_from_scratch.ingest import ingest                               # Day 02's pipeline
from rag_from_scratch.queries import NamesReader, unknown_names_v2       # Day 10's names rule (adopted on Day 12)
from rag_from_scratch.rubric import (COST, Grade, answer_entry, better_or_worse, closest_call,  # Day 12's rubric
                                     compare_answers, grade, value)                             # (new)

print(f"Python {platform.python_version()} | numpy {np.__version__} | onnxruntime {ort.__version__} | "
      f"tokenizers {tokenizers.__version__}")

Python 3.13.18 | numpy 2.4.6 | onnxruntime 1.31.0 | tokenizers 0.23.3


In [2]:
EVALS = REPO_ROOT / "evals"
V4 = {"questions": EVALS / "questions-v4.jsonl", "manifest": EVALS / "manifest-v4.json"}    # Day 12's 266 queries
V5 = {"questions": EVALS / "questions-v5.jsonl", "manifest": EVALS / "manifest-v5.json"}    # + today's 51 new ones
DAY12_BASELINE = EVALS / "baselines" / "day-12-answers-v4.json"     # Day 12's accepted answers: today's reference
BASELINE_FILE = EVALS / "baselines" / "day-13-answers-v5.json"      # today's accepted run
LIVE12 = EVALS / "reviews" / "day-12-live-gpt-6-luna.jsonl"         # Day 12's live pass: replies, tokens, timings
REGRESSIONS_FILE = EVALS / "regressions.jsonl"                      # the series' recorded failures, as test cases
PREREGISTRATION = EVALS / "preregistration" / "day-13.md"           # today's rules, committed before any held-out run
OUT_DIR = REPO_ROOT / "outputs" / "day-13"                          # this run's results and traces (git-ignored)

HITS = TOP_K                  # 3: the weighted fusion's top 3 (Day 11 kept it)
BUDGET = MAX_CONTEXT_WORDS    # 300 words, whole chunks (Day 06)
GATE = 0.62                   # Day 07's measured gate on the dense top score
READER_THRESHOLD = 0.63       # Day 07's measured reader threshold
PIPELINE_VERSION = "day-13.1"  # weighted top 3, dense gate 0.62, Day 10's names rule, reader 0.63, citation check
THETAS = [round(0.80 + i / 100, 2) for i in range(21)]     # the semantic cache's thresholds tried on dev: 0.80 ... 1.00

# Prices in USD per million input / output tokens, OpenAI's standard tier, checked on 2026-11-12
# (https://developers.openai.com/api/docs/pricing). gpt-6-luna is the model Day 12's live pass used.
PRICES = {"gpt-6-luna": (0.10, 0.50), "gpt-6.1-sol": (2.00, 10.00)}
PROMPT_CACHE_MIN_TOKENS = 1024   # OpenAI's prompt caching starts at 1,024 input tokens (GPT-5.6 and later; 2026-11-12)

# The model client's settings (evals/preregistration/day-13.md): real use, and the same logic 20 times faster for the
# local fake servers below
CLIENT = {"attempts": 3, "timeout": 10.0, "deadline": 30.0, "backoff_base": 0.5, "backoff_cap": 8.0}
BREAKER = {"failures": 3, "cooldown": 30.0}
DEMO = {k: (v / 20 if isinstance(v, float) else v) for k, v in CLIENT.items()}
DEMO_BREAKER = {"failures": 3, "cooldown": BREAKER["cooldown"] / 20}

# What evals/preregistration/day-13.md commits to. Sections 5.3 and 7 recompute these on dev and stop if they differ.
PREREGISTERED = {"theta": 0.89, "dev curve": {0.80: (53, 2, 14), 0.85: (16, 0, 4), 0.88: (6, 0, 1), 0.89: (0, 0, 0)},
                 "dev value": {1: 68, 2: 12, 3: -44}, "dev outcomes": (124, 23, 17, 39, 33),
                 "frozen": "2026-11-12 21:11:27 UTC"}
prereg = PREREGISTRATION.read_text(encoding="utf-8")
assert "So **θ = 0.89**" in prereg and "| **0.89** | **0** | 0 | 0 | 0 |" in prereg and \
    "value **+68** at cost 1, **+12** at cost 2, **−44** at cost 3" in prereg and \
    "124 correct / 23 wrong / 17 refused, unanswerable 39\nrefused / 33 answered" in prereg and \
    PREREGISTERED["frozen"] in prereg, "PREREGISTERED must match the pre-registration"

## 1. The same queries, the same snapshot, and Day 12's answers reproduced

`questions-v5` keeps v4's 266 queries exactly (its 76 test questions are dev from today: Day 12 used them to decide) and adds **51 new test questions**, written by a fixed rule after today's rules were frozen: for every 8th answerable dev question a **paraphrase** (same answer) and a **near neighbour** (one detail changed, so the documents give another answer or none), and for every 8th unanswerable one a paraphrase. That makes the test set a stress test for a cache, not a sample of traffic.

Then the reference: Day 12's whole answer baseline, rebuilt with the promoted rubric and compared field by field. From today, the **current pipeline** is the one Day 12 adopted: Day 12's pipeline with Day 10's names rule (`names-v2` in Day 12's file).

In [3]:
manifest4, manifest5 = (json.loads(V["manifest"].read_text(encoding="utf-8")) for V in (V4, V5))
queries4, queries5 = load_queries(V4["questions"]), load_queries(V5["questions"])
documents, _ = ingest(REPO_ROOT)
chunks = [c for d in documents for c in section_chunks(d)]
by_cid = {c.chunk_id: c for c in chunks}
problems = []
for V, manifest, qs in ((V4, manifest4, queries4), (V5, manifest5, queries5)):
    problems += validate(qs, chunks, manifest)
    if hashlib.sha256(V["questions"].read_bytes()).hexdigest() != manifest["questions_sha256"]:
        problems.append(f"{V['questions'].name} changed but its manifest didn't")
by_qid = {q.id: q for q in queries5}
used = [q.id for q in queries4 if q.split == "test"]
# v5 keeps v4's 266 queries exactly (text, user, labels); only the used test questions changed split (and say so)
same266 = all(q5 == dataclasses.replace(q4, split=q5.split, note=q5.note) for q4, q5 in zip(queries4, queries5)) and \
    [q.id for q in queries5[:266] if q.split != "dev"] == [] and \
    all(by_qid[qid].note.endswith("so dev from v5 on)") for qid in used) and \
    all(q5.note == q4.note for q4, q5 in zip(queries4, queries5) if q4.id not in used)
NEW = [q for q in queries5 if q.split == "test"]
natural = [q for q in queries5 if q.kind == "question"]      # an answer is graded for questions, not pasted values
DEV = [q for q in natural if q.split == "dev"]
print(f"{len(queries5)} labelled queries (version {manifest5['version']}): v4's {len(queries4)} "
      f"({'identical, its ' + str(len(used)) + ' test questions now dev' if same266 else 'CHANGED'}) + {len(NEW)} new "
      f"test questions ({sum(q.answerable for q in NEW)} answerable, {sum(not q.answerable for q in NEW)} not), "
      f"same snapshot | label problems: {len(problems)}" + "".join(f"\n  {p}" for p in problems))
print(f"questions to answer end to end: dev {len(DEV)} ({sum(q.answerable for q in DEV)} answerable), "
      f"test {len(NEW)} ({sum(q.answerable for q in NEW)} answerable) | new test questions by type: "
      + ", ".join(f"{t} {n}" for t, n in Counter(q.tags[0] for q in NEW).items()))
assert not problems and same266 and manifest5["snapshot"] == manifest4["snapshot"]
assert {q.origin for q in NEW} == {"day13-new"} and all(q.kind == "question" for q in NEW)

embedder = Embedder(REPO_ROOT)     # Day 04's model: downloaded once (133 MB), SHA-256 checked on every run
index = VectorIndex(embedder.model["repo"], embedder.revision, embedder.dimensions)
sync(index, chunks, embedder.embed)   # the snapshot's chunks, embedded now (Day 05's sync into an empty index)
bm25 = {user: BM25Index({c.chunk_id: c.text for c in chunks if set(c.allowed_groups) & groups})
        for user, groups in DIRECTORY.items()}
INDEX_FINGERPRINT = fingerprint(snapshot(chunks))   # changes whenever a chunk's text or permissions change (Day 07)
rankings = {"BM25": {}, "dense": {}, "weighted": {}}     # retriever -> query id -> full ranking, best first
for q in queries5:
    vector = embedder.embed_query(q.question)
    lexical = [(h.id, h.score) for h in bm25[q.user].search(q.question, len(chunks))]
    dense = [(r.id, r.score) for r in index.search(vector, len(chunks), where=groups_filter(DIRECTORY[q.user]))]
    rankings["BM25"][q.id], rankings["dense"][q.id] = lexical, dense
    rankings["weighted"][q.id] = weighted(lexical, dense, ALPHA)     # the default retriever since Day 08
print(f"BM25, dense and the weighted fusion on {len(queries5)} queries | index fingerprint {INDEX_FINGERPRINT[:16]}")

317 labelled queries (version 5): v4's 266 (identical, its 76 test questions now dev) + 51 new test questions (33 answerable, 18 not), same snapshot | label problems: 0
questions to answer end to end: dev 236 (164 answerable), test 51 (33 answerable) | new test questions by type: paraphrase 30, near-neighbour 21
tokenizer.json       711,396 bytes   SHA-256 d241a60d5e8f... verified


onnx/model.onnx  133,093,490 bytes   SHA-256 828e1496d7fa... verified


BM25, dense and the weighted fusion on 317 queries | index fingerprint 3f8344d00b8b4878


In [4]:
class QueryVectorCache:
    """The question's vector, embedded once per exact text. The dense retriever needs it, and so does the reader
    (it scores every sentence against the question): without this, every request embeds its question twice."""

    def __init__(self, embedder):
        self.embedder, self.model, self.revision, self.vectors = embedder, embedder.model, embedder.revision, {}
        self.hits = self.misses = 0

    def embed(self, texts, *args, **kwargs):
        return self.embedder.embed(texts, *args, **kwargs)

    def embed_query(self, question: str, instruction: str = QUERY_INSTRUCTION):
        key = (self.revision, instruction, question)          # everything the vector depends on
        if key in self.vectors:
            self.hits += 1
        else:
            self.misses += 1
            self.vectors[key] = self.embedder.embed_query(question, instruction)
        return self.vectors[key]


query_vectors = QueryVectorCache(embedder)
sentences = SentenceCache(query_vectors)          # Day 11: each sentence's vector by its exact text
READERS = {"current": ExtractiveReader(sentences, min_score=READER_THRESHOLD),             # Day 06's names rule
           "names-v2": NamesReader(sentences, unknown_names_v2, min_score=READER_THRESHOLD)}  # Day 10's (Day 12's D1)
READER = READERS["names-v2"]                      # the current pipeline from today on


def sent_of(q) -> tuple[str, ...]:
    return tuple(cid for cid, _ in rankings["weighted"][q.id][:HITS])        # the current pipeline's context


def cutoff_gap(q, n: int = HITS) -> float:
    # How close the fused scores at ranks n and n + 1 are: under TIE, another CPU's rounding could swap them
    ranked = rankings["weighted"][q.id]
    return ranked[n - 1][1] - ranked[n][1] if n < len(ranked) else math.inf


def run(q, generator=READER):
    # The current pipeline, end to end, untraced (Day 11's answer_context(): gate, permission check, budget, reader,
    # citation check)
    return answer_context(q.user, q.question, sent_of(q), by_cid, dict(rankings["weighted"][q.id]),
                          rankings["dense"][q.id][0][1], generator, gate=GATE)


# Day 12's accepted answer baseline, rebuilt: every answer, verdict, closest call and metric of both of its pipelines,
# with its settings, decisions and reviews taken from the file (Day 12's notebook test checks those on every push)
day12 = json.loads(DAY12_BASELINE.read_text(encoding="utf-8"))
PIPELINES12 = tuple(day12["pipelines"])                          # ("current", "names-v2")
v4_natural = [q for q in queries4 if q.kind == "question"]
ANSWERS12 = {p: {q.id: run(q, READERS[p]) for q in v4_natural} for p in PIPELINES12}
GRADES12 = {p: {q.id: grade(ANSWERS12[p][q.id], q, sent_of(q)) for q in v4_natural} for p in PIPELINES12}


def counts(grades: dict, qs, cost: float = COST) -> tuple:
    # answerable correct / wrong / refused, unanswerable refused / answered, and the total value at this cost
    a = Counter(grades[q.id].outcome for q in qs if q.answerable)
    u = Counter(grades[q.id].outcome for q in qs if not q.answerable)
    return (a["correct"], a["wrong"], a["refused"], u["refused"], u["answered"],
            sum(value(grades[q.id].outcome, cost) for q in qs))


METRICS = ("correct", "wrong", "refused", "unanswerable refused", "unanswerable answered", f"value at cost {COST}")
reproduced12 = {**day12,
                "questions": {q.id: {"split": q.split,
                                     "retrieval": {"top3": list(sent_of(q)),
                                                   "dense_top": round(rankings["dense"][q.id][0][1], 4),
                                                   "first_relevant": first_relevant(rankings["weighted"][q.id],
                                                                                    q.answer_chunks)
                                                   if q.answerable else None,
                                                   "margin": round(cutoff_gap(q), 4)},
                                     **{p: answer_entry(ANSWERS12[p][q.id], GRADES12[p][q.id],
                                                        closest_call(ANSWERS12[p][q.id], READERS[p],
                                                                     rankings["dense"][q.id][0][1], GATE, cutoff_gap(q)))
                                        for p in PIPELINES12}} for q in v4_natural},
                "metrics": {label: {p: dict(zip(METRICS, counts(GRADES12[p], [q for q in v4_natural if q.split == s])))
                                    for p in PIPELINES12} for label, s in (("dev", "dev"), ("v4 test", "test"))}}
changes12, ties12 = compare_answers(reproduced12, day12, PIPELINES12)
same_retrieval = all(reproduced12["questions"][qid]["retrieval"] == day12["questions"][qid]["retrieval"]
                     for qid in day12["questions"])
print(f"{len(v4_natural)} v4 questions x {len(PIPELINES12)} pipelines compared with "
      f"{DAY12_BASELINE.relative_to(REPO_ROOT).as_posix()}: "
      + (f"{len(changes12)} changes" + "".join(f"\n  {c}" for c in changes12) if changes12 else
         "no changes (status, answer text, citations, every field of the verdict, metrics; contexts sent "
         + ("identical)" if same_retrieval else "DIFFERENT)")))
for t in ties12:
    print(f"  near tie, not a change: {t}")
regressions = [json.loads(line) for line in REGRESSIONS_FILE.read_text(encoding="utf-8").splitlines() if line.strip()]
print(f"{len(regressions)} regression cases in {REGRESSIONS_FILE.relative_to(REPO_ROOT).as_posix()} (tests/"
      f"test_regressions.py runs every one): " + ", ".join(f"{n} {s}" for s, n in
                                                        Counter(r['status'] for r in regressions).most_common()))

236 v4 questions x 2 pipelines compared with evals/baselines/day-12-answers-v4.json: no changes (status, answer text, citations, every field of the verdict, metrics; contexts sent identical)
33 regression cases in evals/regressions.jsonl (tests/test_regressions.py runs every one): 13 guard, 11 known, 9 fixed


## 2. A trace for every request

A **trace** is the record of one request: a list of **spans**, one per step, each with its start, its duration and what the step produced or decided. When an answer is wrong, slow or expensive, the trace says which step to look at.

What goes in, and what never does:

| In the trace | Never in the trace |
|---|---|
| the user id and groups, the question's SHA-256 and length, the index fingerprint, the pipeline version | the question's text (it may hold personal data) |
| per step: ids and scores (top 3 of each retriever, the sections sent), decisions (gate, names rule, reader, check), token counts or estimates, timings | any document text: retrieved chunks are as sensitive as the documents |
| for a model call: provider, model, attempt, outcome, tokens | the key, the prompt, the reply |

The names follow OpenTelemetry's GenAI conventions where one exists (`gen_ai.operation.name`, `gen_ai.usage.input_tokens`, `error.type`); those conventions are still marked *Development*, and they also leave prompts, replies and retrieved documents out by default. `traced_answer()` is the current pipeline written step by step, calling the same shared functions as `answer_context()` in the same order, each step inside a span. One request, with nothing cached yet:

In [5]:
def sha16(text: str) -> str:
    return hashlib.sha256(text.encode("utf-8")).hexdigest()[:16]


class Trace:
    """One request's trace: spans with a name, a start and a duration (ms) and what the step decided. Ids, scores,
    decisions, timings and token counts only: never a document's text, the question's text or a key."""

    def __init__(self, request_id: str, **attributes):
        self.request_id, self.attributes, self.spans = request_id, dict(attributes), []
        self.started = time.perf_counter()

    def elapsed_ms(self) -> float:
        return (time.perf_counter() - self.started) * 1000

    @contextmanager
    def span(self, name: str, **attributes):
        record = {"name": name, "start_ms": round(self.elapsed_ms(), 3), **attributes}
        began = time.perf_counter()
        try:
            yield record                                    # the step writes what it decided into its span
        except Exception as error:
            record["error.type"] = type(error).__name__     # OpenTelemetry's attribute name for a failure
            raise
        finally:
            record["ms"] = round((time.perf_counter() - began) * 1000, 3)
            self.spans.append(record)

    def finish(self, **attributes) -> dict:
        self.attributes.update(attributes, ms=round(self.elapsed_ms(), 3))
        return {"request_id": self.request_id, **self.attributes,
                "spans": sorted(self.spans, key=lambda s: s["start_ms"])}


@dataclass
class World:
    """What a request reads: the chunk records, the vector index, a BM25 index per user, and the index's fingerprint."""
    by_cid: dict
    index: VectorIndex
    bm25: dict
    fingerprint: str


WORLD = World(by_cid, index, bm25, INDEX_FINGERPRINT)


def top(ranked, n: int = 3) -> list:
    return [[cid, round(score, 4)] for cid, score in ranked[:n]]


def tokens_estimate(text: str) -> int:
    return math.ceil(len(text) / 4)           # rule of thumb: about 4 characters per token (checked in section 4)


def reader_decision(note: str) -> dict:
    # What the reader decided, without the names it found (they are words of the question)
    if note.startswith("no source mentions"):
        return {"decision": "names rule", "names": len(note.removeprefix("no source mentions ").split(", "))}
    found_score = re.match(r"best sentence scores ([0-9.]+) (>=|<) ", note)
    if found_score:
        return {"decision": "answer" if found_score.group(2) == ">=" else "reader threshold",
                "best": float(found_score.group(1))}
    return {"decision": "model" if not note else note.split(":")[0][:40]}


def problem_kind(problem: str) -> str:
    return ("no citation" if problem == "no citation" else "unknown source" if "was not among" in problem
            else "number not in the cited source" if "is not in the cited source" in problem else "other")


def traced_answer(world: World, user: str, question: str, generator, *, queries=None, answers=None,
                  request_id: str = "req") -> tuple[RagAnswer, dict]:
    # The current pipeline, step by step, each step in its own span. It calls the same shared functions as
    # answer_context() in the same order, so the answer is identical (checked below for every question).
    groups = DIRECTORY.get(user, set())
    trace = Trace(request_id, user=user, groups=sorted(groups), question_sha256=sha16(question),
                  question_words=len(question.split()), index=world.fingerprint[:16], pipeline=PIPELINE_VERSION)
    if answers is not None:                                       # an exact answer cache (section 5)
        with trace.span("cache.answers") as s:
            cached = answers.get(user, question, world.fingerprint)
            s["hit"] = cached is not None
        if cached is not None:
            return cached, trace.finish(status=cached.status, served_by="answer cache", answer_sha256=sha16(cached.text),
                                        cited=cited_ids(cached))
    with trace.span("embed_query") as s:
        if queries is not None:
            hits_before = queries.hits
            vector = queries.embed_query(question)
            s["cache_hit"] = queries.hits > hits_before
        else:
            vector = embedder.embed_query(question)
            s["cache_hit"] = False
    with trace.span("retrieve.bm25") as s:
        lexical = [(h.id, h.score) for h in world.bm25[user].search(question, len(world.by_cid))]
        s.update(candidates=len(lexical), top=top(lexical))
    with trace.span("retrieve.dense") as s:
        dense = [(r.id, r.score) for r in world.index.search(vector, len(world.by_cid), where=groups_filter(groups))]
        s.update(candidates=len(dense), top=top(dense))
    with trace.span("retrieve.fuse", **{"gen_ai.operation.name": "retrieval", "gen_ai.retrieval.top_k": HITS}) as s:
        fused = weighted(lexical, dense, ALPHA)
        sent = tuple(cid for cid, _ in fused[:HITS])
        s.update(documents=top(fused, HITS), gap_after_last=round(fused[HITS - 1][1] - fused[HITS][1], 4))
        if any(not set(world.by_cid[cid].allowed_groups) & groups for cid in sent):     # checked again before use
            raise PermissionError(f"a context holds a chunk {user} may not read")
    with trace.span("gate", threshold=GATE) as s:
        dense_top = dense[0][1]
        passed = dense_top >= GATE
        s.update(dense_top=round(dense_top, 4), decision="pass" if passed else "refuse")
    if not passed:
        a = RagAnswer(user, question, "abstained", ABSTAIN, "below the dense gate: no generator called")
    else:
        with trace.span("context", budget_words=BUDGET) as s:
            scores = dict(fused)
            context = assemble(question, [Result(i, scores[cid], cid, world.by_cid[cid]) for i, cid in enumerate(sent, 1)],
                               BUDGET)
            s.update(sent=[x.id for x in context.sources], dropped=list(context.dropped),
                     words=sum(count_words(x.text) for x in context.sources),
                     prompt_tokens_estimate=sum(tokens_estimate(m["content"]) for m in build_messages(context)))
        with trace.span("generate") as s:
            reply = generator.generate(context, trace) if getattr(generator, "traced", False) else \
                generator.generate(context)
            s.update(generator=reply.generator, **reader_decision(reply.note))
            if reply.usage:
                s.update({"gen_ai.usage.input_tokens": reply.usage.get("input_tokens"),
                          "gen_ai.usage.output_tokens": reply.usage.get("output_tokens")})
        with trace.span("check") as s:
            check = check_citations(reply.text, context)
            status = "abstained" if check.abstained else "blocked" if not check.passed else "answered"
            s.update(claims=len(check.claims), problems=[problem_kind(p) for p in check.problems], decision=status)
        a = RagAnswer(user, question, status, {"abstained": ABSTAIN, "blocked": BLOCKED}.get(status, reply.text),
                      reply.note, context, check, reply)
    if answers is not None:
        answers.put(user, question, world.fingerprint, a)
    return a, trace.finish(status=a.status, served_by=a.reply.generator if a.reply else "gate",
                           answer_sha256=sha16(a.text), cited=cited_ids(a))


def cited_ids(a: RagAnswer) -> list[str]:
    return list(dict.fromkeys(i for c in a.check.claims for i in c.cited)) if a.check else []


def brief(value):
    # A compact view of a span's attribute: a ranking shows its first (id, score) and how many more
    if isinstance(value, list) and value and isinstance(value[0], list):
        return f"{value[0][0]} {value[0][1]}" + (f" (+{len(value) - 1})" if len(value) > 1 else "")
    if isinstance(value, list):
        return ", ".join(map(str, value)) or "-"
    return value


def show_trace(t: dict) -> None:
    # A waterfall: each step's start, its duration as a bar, and what it decided
    print(f"{t['request_id']}  user {t['user']} {t['groups']}  question sha256 {t['question_sha256']} "
          f"({t['question_words']} words)  index {t['index']}  pipeline {t['pipeline']}")
    scale = max(t["ms"], 1e-9)
    for s in t["spans"]:
        what = "; ".join(f"{k} {brief(v)}" for k, v in s.items() if k not in ("name", "start_ms", "ms", "parent"))
        bar = "#" * max(1, round(24 * s["ms"] / scale))
        name = ("  " + s["name"]) if "parent" in s else s["name"]            # a step inside another one
        print(f"  {name:<18}{s['start_ms']:>7.1f} {s['ms']:>7.1f} ms {bar:<25}{what}")
    print(f"  total {t['ms']:.1f} ms -> {t['status']}, served by {t['served_by']}, cites {', '.join(t['cited']) or '-'}")


LOST = by_qid["d03-13"]                 # "How soon must I report a lost laptop?" (Day 03's probe)
cold_reader = NamesReader(SentenceCache(embedder), unknown_names_v2, min_score=READER_THRESHOLD)  # nothing cached yet
answer, trace = traced_answer(WORLD, LOST.user, LOST.question, cold_reader, request_id="req-0001")
show_trace(trace)
print(f"labelled answer chunk: {', '.join(LOST.answer_chunks)} | outcome: {outcome(answer, LOST)}")

req-0001  user alice ['all-employees']  question sha256 e5a84cae394b1030 (8 words)  index 3f8344d00b8b4878  pipeline day-13.1
  embed_query           0.0    15.0 ms ###                      cache_hit False
  retrieve.bm25        15.0     0.1 ms #                        candidates 11; top travel-policy#safety-while-travelling 7.2761 (+2)
  retrieve.dense       15.1     0.2 ms #                        candidates 43; top security-incident-reporting#what-not-to-do 0.6758 (+2)
  retrieve.fuse        15.3     0.1 ms #                        gen_ai.operation.name retrieval; gen_ai.retrieval.top_k 3; documents travel-policy#safety-while-travelling 0.9485 (+2); gap_after_last 0.0453
  gate                 15.4     0.0 ms #                        threshold 0.62; dense_top 0.6758; decision pass
  context              15.4     0.1 ms #                        budget_words 300; sent travel-policy#safety-while-travelling, security-incident-reporting#what-not-to-do, security-incident-reporting#how-to-

The trace shows the lost-laptop failure without a word of text: *How to report*, the labelled section, was sent (third), and the answer cites *Safety while travelling*. That is Day 12's stage "reader: answer in the context, not used", read straight from ids. It also shows where the time went: the reader embedded every sentence it read, and the question a second time.

Every question is traced the same way, and the instrumented pipeline must give exactly the untraced pipeline's answer:

In [6]:
# Every question, traced: the instrumented pipeline must give exactly the answer of the untraced one
TRACES = []
for i, q in enumerate(natural, start=1):
    a, t = traced_answer(WORLD, q.user, q.question, READER, queries=query_vectors, request_id=f"req-{i:04d}")
    assert dataclasses.asdict(a) == dataclasses.asdict(run(q)), f"tracing changed the answer to {q.id}"
    TRACES.append(t)

# What a trace must never hold: a run of six words from any chunk, four words in a row from the question, a key
CHUNK_RUNS = {" ".join(w[i:i + 6]) for c in chunks for w in [c.text.lower().split()] for i in range(len(w) - 5)}


def strings(value) -> list[str]:
    if isinstance(value, dict):
        return [s for k, v in value.items() for s in [k, *strings(v)]]
    if isinstance(value, (list, tuple)):
        return [s for v in value for s in strings(v)]
    return [value] if isinstance(value, str) else []


def names_in(question: str) -> set[str]:
    """Capitalised words after the first (Chrome, Madrid, ...): a single one can identify the request."""
    words = [w.strip("?.,!:;'\"()") for w in question.split()[1:]]
    return {w.lower() for w in words if w[:1].isupper() and len(w) > 1}


def leaks(t: dict, question: str, secrets=()) -> list[str]:
    words = " ".join(strings(t)).lower().split()
    grams6 = {" ".join(words[i:i + 6]) for i in range(len(words) - 5)}
    grams4 = {" ".join(words[i:i + 4]) for i in range(len(words) - 3)}
    q = question.lower().split()
    found = [f"document text: {g!r}" for g in sorted(grams6 & CHUNK_RUNS)]
    found += [f"question text: {g!r}" for g in sorted(grams4 & {" ".join(q[i:i + 4]) for i in range(len(q) - 3)})]
    found += [f"a name from the question: {w!r}" for w in sorted(names_in(question) &
                                                                 {x.strip("?.,!:;'\"()") for x in words})]
    found += [f"a secret ({s[:4]}...)" for s in secrets if s and s in json.dumps(t)]
    return found


trace_leaks = [(q.id, x) for q, t in zip(natural, TRACES) for x in leaks(t, q.question)]
OUT_DIR.mkdir(parents=True, exist_ok=True)
with (OUT_DIR / "traces.jsonl").open("w", encoding="utf-8", newline="\n") as f:
    for t in TRACES:
        f.write(json.dumps(t, ensure_ascii=False) + "\n")
size = (OUT_DIR / "traces.jsonl").stat().st_size
print(f"{len(TRACES)} questions traced: every answer identical to the untraced pipeline's | spans per trace: "
      f"{min(len(t['spans']) for t in TRACES)} to {max(len(t['spans']) for t in TRACES)} | document text, question "
      f"text or keys found: {len(trace_leaks)} | written to outputs/day-13/traces.jsonl ({size / len(TRACES):.0f} "
      f"bytes per request)")
by_status = Counter(t["status"] for t in TRACES)
print("traced outcomes: " + ", ".join(f"{s} {n}" for s, n in by_status.most_common()) + " | refused at the gate "
      f"(no reader called): {sum(any(s['name'] == 'gate' and s['decision'] == 'refuse' for s in t['spans']) for t in TRACES)}")
# A trace answers "which step lost the answer?" without any text: the labelled section was sent, another was cited
not_used = [q.id for q, t in zip(natural, TRACES) if q.answerable and t["status"] == "answered" and
            set(q.answer_chunks) & set(next(s["sent"] for s in t["spans"] if s["name"] == "context")) and
            not set(q.answer_chunks) & set(t["cited"])]
print(f"answerable questions whose trace shows the labelled section sent and another one cited: {len(not_used)} "
      f"({', '.join(not_used[:6])}, ...)")

287 questions traced: every answer identical to the untraced pipeline's | spans per trace: 5 to 8 | document text, question text or keys found: 0 | written to outputs/day-13/traces.jsonl (1779 bytes per request)
traced outcomes: answered 219, abstained 63, blocked 5 | refused at the gate (no reader called): 39
answerable questions whose trace shows the labelled section sent and another one cited: 18 (d03-13, d01-01, d07-a08, d10-a27, d11-a08, d11-a29, ...)


## 3. Latency: p50 and p95 per step

**p50** (the median) is a typical request; **p95** is the time 95% of requests stay under, the one users notice. Percentiles here use the nearest-rank method on this machine's timings; they vary between runs and machines, so no test pins them.

Three ways to run the same pipeline: **cold** (nothing cached: each request embeds every sentence it reads, and its question twice), **warm** (every sentence vector computed once when the documents are indexed, and the question embedded once per request) and **repeated** (the same user asks the same question again: the answer cache, section 5).

In [7]:
class AnswerCache:
    """Whole answers, keyed on everything an answer depends on: who asks (their groups, not their name), the exact
    question, which documents (the index fingerprint) and which pipeline (its version). Before serving, every chunk the
    answer cites is checked against the asker's groups again: a lookup by key isn't a search (Day 11)."""

    def __init__(self, version: str, records):
        self.version, self.records, self.entries = version, records, {}
        self.hits = self.misses = 0

    def key(self, user: str, question: str, index_fingerprint: str) -> tuple:
        return (tuple(sorted(DIRECTORY.get(user, ()))), question, index_fingerprint, self.version)

    def get(self, user: str, question: str, index_fingerprint: str):
        a = self.entries.get(self.key(user, question, index_fingerprint))
        if a is not None and all(may_read(cid, user, self.records) for cid in cited_ids(a)):
            self.hits += 1
            return a
        self.misses += 1
        return None

    def put(self, user: str, question: str, index_fingerprint: str, a: RagAnswer) -> None:
        if a.status != "unavailable":                     # a failure isn't an answer: never cache it
            self.entries[self.key(user, question, index_fingerprint)] = a


def nearest_rank(values, p: float) -> float:
    # The p-th percentile by the nearest-rank method: the smallest value with at least p% of the values at or below it
    ordered = sorted(values)
    return ordered[max(0, math.ceil(p / 100 * len(ordered)) - 1)]


STEPS = ["cache.answers", "embed_query", "retrieve.bm25", "retrieve.dense", "retrieve.fuse", "gate", "context",
         "generate", "check"]


def latency_table(traces: list[dict], label: str) -> dict:
    totals = [t["ms"] for t in traces]
    print(f"{label}: {len(traces)} requests | whole request p50 {nearest_rank(totals, 50):.2f} ms, "
          f"p95 {nearest_rank(totals, 95):.2f} ms")
    rows = {}
    for step in STEPS:
        ms = [s["ms"] for t in traces for s in t["spans"] if s["name"] == step]
        if ms:
            rows[step] = (len(ms), nearest_rank(ms, 50), nearest_rank(ms, 95), sum(ms) / sum(totals))
            print(f"  {step:<15}{len(ms):>5} spans   p50 {rows[step][1]:>8.3f} ms   p95 {rows[step][2]:>8.3f} ms   "
                  f"{rows[step][3]:>4.0%} of all the time")
    print(f"  the steps' p95s add up to {sum(r[2] for r in rows.values()):.2f} ms; the requests' p95 is "
          f"{nearest_rank(totals, 95):.2f} ms (percentiles don't add up)")
    return {"p50": nearest_rank(totals, 50), "p95": nearest_rank(totals, 95), "steps": rows}


print(f"on this machine: {os.cpu_count()} CPUs, {platform.system()}, Python {platform.python_version()} "
      "(timings vary between runs and machines: a description, not pinned by any test)\n")
SAMPLE = DEV[::4]       # 59 dev questions: each one alone, with nothing cached, costs about a tenth of a second
cold = []
for i, q in enumerate(SAMPLE, start=1):
    fresh = NamesReader(SentenceCache(embedder), unknown_names_v2, min_score=READER_THRESHOLD)
    cold.append(traced_answer(WORLD, q.user, q.question, fresh, request_id=f"cold-{i:04d}")[1])
LATENCY = {"cold": latency_table(cold, "Cold (nothing cached: each request embeds its question twice and every "
                                       "sentence it reads)")}
print()

# Warm: what the sentence vectors depend on is known when the documents are indexed, so embed them all then
warm_queries = QueryVectorCache(embedder)
warm_sentences = SentenceCache(warm_queries)
started = time.perf_counter()
units = [f"{c.title} > {c.section}: {u}" for c in chunks
         for u in answer_units(Source(c.chunk_id, c.title, c.section, c.last_reviewed, c.text, 0.0, c.source_path))]
warm_sentences.embed(units)
precompute_s = time.perf_counter() - started
warm_reader = NamesReader(warm_sentences, unknown_names_v2, min_score=READER_THRESHOLD)
answers_cache = AnswerCache(PIPELINE_VERSION, by_cid)
warm = [traced_answer(WORLD, q.user, q.question, warm_reader, queries=warm_queries, answers=answers_cache,
                      request_id=f"warm-{i:04d}")[1] for i, q in enumerate(DEV, start=1)]
print(f"precomputed at index time: {len(units)} sentence and table-row vectors in {precompute_s:.2f} s (once)")
LATENCY["warm"] = latency_table(warm, "Warm (sentence vectors precomputed; the question embedded once per request; "
                                      "first time each question is asked)")
print()
repeat = [traced_answer(WORLD, q.user, q.question, warm_reader, queries=warm_queries, answers=answers_cache,
                        request_id=f"again-{i:04d}")[1] for i, q in enumerate(DEV, start=1)]
LATENCY["repeat"] = latency_table(repeat, "Repeated (the same user asks the same question again: the answer cache)")
assert all(t["served_by"] == "answer cache" for t in repeat) and answers_cache.hits == len(DEV)
print()

live12 = [json.loads(line) for line in LIVE12.read_text(encoding="utf-8").splitlines() if line.strip()]
seconds = [r["seconds"] for r in live12]
print(f"The model call, recorded by Day 12's live pass ({len(live12)} requests to gpt-6-luna from the user's PC, "
      f"2026-11-12; context, model and check): p50 {nearest_rank(seconds, 50):.2f} s, p95 "
      f"{nearest_rank(seconds, 95):.2f} s, slowest {max(seconds):.2f} s: about "
      f"{nearest_rank(seconds, 50) * 1000 / LATENCY['warm']['p50']:.0f} times the warm API-free request's p50")

on this machine: 2 CPUs, Linux, Python 3.13.18 (timings vary between runs and machines: a description, not pinned by any test)



Cold (nothing cached: each request embeds its question twice and every sentence it reads): 59 requests | whole request p50 119.56 ms, p95 167.05 ms
  embed_query       59 spans   p50   14.550 ms   p95   18.614 ms    14% of all the time
  retrieve.bm25     59 spans   p50    0.102 ms   p95    0.158 ms     0% of all the time
  retrieve.dense    59 spans   p50    0.183 ms   p95    0.255 ms     0% of all the time
  retrieve.fuse     59 spans   p50    0.054 ms   p95    0.096 ms     0% of all the time
  gate              59 spans   p50    0.002 ms   p95    0.002 ms     0% of all the time
  context           50 spans   p50    0.067 ms   p95    0.113 ms     0% of all the time
  generate          50 spans   p50  107.953 ms   p95  151.995 ms    86% of all the time
  check             50 spans   p50    0.057 ms   p95    0.092 ms     0% of all the time
  the steps' p95s add up to 171.33 ms; the requests' p95 is 167.05 ms (percentiles don't add up)



precomputed at index time: 102 sentence and table-row vectors in 1.57 s (once)
Warm (sentence vectors precomputed; the question embedded once per request; first time each question is asked): 236 requests | whole request p50 12.10 ms, p95 16.77 ms
  cache.answers    236 spans   p50    0.002 ms   p95    0.003 ms     0% of all the time
  embed_query      236 spans   p50   11.597 ms   p95   15.992 ms    96% of all the time
  retrieve.bm25    236 spans   p50    0.074 ms   p95    0.123 ms     1% of all the time
  retrieve.dense   236 spans   p50    0.124 ms   p95    0.196 ms     1% of all the time
  retrieve.fuse    236 spans   p50    0.044 ms   p95    0.081 ms     0% of all the time
  gate             236 spans   p50    0.001 ms   p95    0.002 ms     0% of all the time
  context          207 spans   p50    0.052 ms   p95    0.090 ms     0% of all the time
  generate         207 spans   p50    0.103 ms   p95    0.169 ms     1% of all the time
  check            207 spans   p50    0.033 ms   

On the API-free path, a cold request spends most of its time in the reader, embedding sentences that don't change between requests. Computing them **when the documents are indexed** (about a hundred vectors, about a second and a half, once) moves that work off the request path: a warm request then spends almost all of its time embedding the question. A repeated question costs microseconds. And a hosted model makes all of this small: Day 12's live calls took about a second each, tens of times the whole API-free request.

The steps' p95s and the requests' p95 are close here only because one step dominates each request; in general the slowest requests aren't slow at every step, so add the timings per request first, then take the percentile.

## 4. Cost: where the tokens go

A hosted model is paid per token: input (the instructions, the sources, the question) and output (the answer, and the model's reasoning, which is billed as output). Day 12's live pass recorded the provider's own count for 40 requests, so the usual estimate (about four characters per token) can be checked against it.

In [8]:
def prompt_parts(context) -> dict:
    # Day 06's two messages, split into what they are made of
    instructions, user = (m["content"] for m in build_messages(context))
    question = f"\n\nQuestion: {context.question}"
    return {"instructions": instructions, "sources": user[:-len(question)], "question": question}


# Day 12's 40 live requests: the same contexts, rebuilt here, against the tokens the provider counted
rows = []
for r in live12:
    q = by_qid[r["id"]]
    context = run(q).context                      # the current pipeline's context for it (all 40 passed the gate)
    parts = {k: tokens_estimate(v) for k, v in prompt_parts(context).items()}
    rows.append((r, parts, sum(parts.values())))
ratios = [r["usage"]["input_tokens"] / est for r, parts, est in rows]
spent_in = sum(r["usage"]["input_tokens"] for r, _, _ in rows)
spent_out = sum(r["usage"]["output_tokens"] for r, _, _ in rows)
mean_in, mean_out = spent_in / len(rows), spent_out / len(rows)
print(f"Day 12's live pass: {len(rows)} requests, {spent_in} input and {spent_out} output tokens counted by the provider "
      f"(mean {mean_in:.0f} in, {mean_out:.1f} out; output from {min(r['usage']['output_tokens'] for r, _, _ in rows)} "
      f"for 'I don't know' to {max(r['usage']['output_tokens'] for r, _, _ in rows)}, reasoning included)")
print(f"estimate (characters / 4) against the provider's count of input tokens: actual / estimate "
      f"{statistics.mean(ratios):.2f} on average ({min(ratios):.2f} to {max(ratios):.2f})")
shares = {k: sum(p[k] for _, p, _ in rows) / sum(e for _, _, e in rows) for k in ("instructions", "sources", "question")}
print("where the input tokens go (estimated): " + ", ".join(f"{k} {v:.0%}" for k, v in shares.items())
      + f" | the longest prompt: about {max(e for _, _, e in rows)} tokens, under the {PROMPT_CACHE_MIN_TOKENS} the "
      "provider's prompt cache needs")
print()
COST_PER_REQUEST = {}
for model, (price_in, price_out) in PRICES.items():
    cost_in, cost_out = mean_in * price_in / 1e6, mean_out * price_out / 1e6
    COST_PER_REQUEST[model] = cost_in + cost_out
    print(f"{model:<12} USD {price_in:.2f} / {price_out:.2f} per million: USD {cost_in + cost_out:.6f} per request "
          f"(input {cost_in / (cost_in + cost_out):.0%}) = USD {1000 * (cost_in + cost_out):.3f} per 1,000 requests "
          "that reach the model")
recorded = (spent_in * PRICES["gpt-6-luna"][0] + spent_out * PRICES["gpt-6-luna"][1]) / 1e6
print(f"Day 12's 40 requests at these prices: USD {recorded:.5f}")
print()
# What doesn't reach the model: the gate refuses before any generator is called (Day 07); a repeated question is served
# from the answer cache (section 5); the API-free reader costs CPU time, not tokens
gated = [t for t in warm if any(s["name"] == "gate" and s["decision"] == "refuse" for s in t["spans"])]
reach = len(DEV) - len(gated)
print(f"of the {len(DEV)} dev questions, the gate refuses {len(gated)} without calling a generator, so {reach} would "
      f"reach a model: USD {reach * COST_PER_REQUEST['gpt-6-luna']:.4f} with gpt-6-luna, "
      f"USD {reach * COST_PER_REQUEST['gpt-6.1-sol']:.4f} with gpt-6.1-sol for all of them; asked again, "
      f"{answers_cache.hits} of {len(DEV)} come from the answer cache and cost nothing")
cpu_ms = sum(t["ms"] for t in warm) / len(warm)
print(f"the API-free path instead: about {cpu_ms:.0f} ms of this machine's CPU per request (warm), "
      f"{len(DEV) * cpu_ms / 1000:.1f} s for all {len(DEV)}")

Day 12's live pass: 40 requests, 18075 input and 1133 output tokens counted by the provider (mean 452 in, 28.3 out; output from 12 for 'I don't know' to 72, reasoning included)
estimate (characters / 4) against the provider's count of input tokens: actual / estimate 0.91 on average (0.87 to 0.97)
where the input tokens go (estimated): instructions 38%, sources 58%, question 4% | the longest prompt: about 543 tokens, under the 1024 the provider's prompt cache needs

gpt-6-luna   USD 0.10 / 0.50 per million: USD 0.000059 per request (input 76%) = USD 0.059 per 1,000 requests that reach the model
gpt-6.1-sol  USD 2.00 / 10.00 per million: USD 0.001187 per request (input 76%) = USD 1.187 per 1,000 requests that reach the model
Day 12's 40 requests at these prices: USD 0.00237

of the 236 dev questions, the gate refuses 29 without calling a generator, so 207 would reach a model: USD 0.0123 with gpt-6-luna, USD 0.2457 with gpt-6.1-sol for all of them; asked again, 236 of 236 come from the an

The estimate **over-counts by about a tenth** on these prompts, a safe direction for a budget. More than half of the input is the sources, and more than a third is the same instructions on every request: the provider's prompt cache would bill those repeated tokens at a tenth of the input price (a twentieth for gpt-6.1-sol), but only from 1,024 input tokens, and these prompts are about half that. Per 1,000 requests that reach the model, gpt-6-luna costs about six cents; gpt-6.1-sol, twenty times more. The gate already keeps 29 of the 236 dev questions away from any model.

## 5. Caching

A cache reuses a result instead of computing it again. It is safe when its **key holds everything the result depends on**: change any of it and the key changes. The two exact caches used above, keyed that way:

| Cache | Key | Depends on |
|---|---|---|
| question vectors | the model revision, the query instruction, the exact question | nothing else: the same text gives the same vector |
| sentence vectors (Day 11's) | the model revision, the exact sentence | nothing else |
| answers | the asker's **groups**, the exact question, the **index fingerprint**, the pipeline's **version** | who may read what, which documents, which code and settings |

Before serving a cached answer, every chunk it cites is checked against the asker's groups again (Day 11's rule: a lookup by key isn't a search).

In [9]:
# 5.1 The exact caches change no answer: every v5 question answered warm (query vectors, precomputed sentence vectors)
# into an empty answer cache, then again from it, against the untraced pipeline
exact = AnswerCache(PIPELINE_VERSION, by_cid)
embedded_before, reused_before = warm_queries.misses, warm_queries.hits
for q in natural:
    first, _ = traced_answer(WORLD, q.user, q.question, warm_reader, queries=warm_queries, answers=exact)
    again, t = traced_answer(WORLD, q.user, q.question, warm_reader, queries=warm_queries, answers=exact)
    assert dataclasses.asdict(first) == dataclasses.asdict(again) == dataclasses.asdict(run(q)), q.id
    assert t["served_by"] == "answer cache"
print(f"exact caches: {len(natural)} questions answered with and without them: identical answers | the first time, "
      f"{exact.misses} answers computed ({warm_queries.misses - embedded_before} questions embedded; every other question "
      f"vector, {warm_queries.hits - reused_before} times, came from the query-vector cache); "
      f"the second time, {exact.hits} answers from the cache")
print()


# 5.2 Two keys that look reasonable and aren't
class QuestionOnlyCache(AnswerCache):
    """A key without the asker: one shared answer per question, and no permission check when serving it."""

    def key(self, user, question, index_fingerprint):
        return (question, index_fingerprint, self.version)

    def get(self, user, question, index_fingerprint):
        return self.entries.get(self.key(user, question, index_fingerprint))


class NoFingerprintCache(AnswerCache):
    """A key without the documents: the same answer, whatever the index holds now."""

    def key(self, user, question, index_fingerprint):
        return (tuple(sorted(DIRECTORY.get(user, ()))), question, self.version)


PAY_MARIA, PAY_ALICE = by_qid["d06-01"], by_qid["d06-02"]      # "When do pay rises take effect?", maria then alice
assert PAY_MARIA.question == PAY_ALICE.question
for cache in (QuestionOnlyCache(PIPELINE_VERSION, by_cid), AnswerCache(PIPELINE_VERSION, by_cid)):
    maria_answer, _ = traced_answer(WORLD, "maria", PAY_MARIA.question, warm_reader, answers=cache)
    alice_answer, t = traced_answer(WORLD, "alice", PAY_ALICE.question, warm_reader, answers=cache)
    leaked = [cid for cid in cited_ids(alice_answer) if not may_read(cid, "alice", by_cid)]
    print(f"{type(cache).__name__}: maria asks, then alice asks the same question -> alice gets "
          f"[{alice_answer.status}] {alice_answer.text[:70]}... (served by {t['served_by']})"
          + (f"\n  LEAK: it cites {', '.join(leaked)}, which only managers may read" if leaked else ""))
    if isinstance(cache, QuestionOnlyCache):
        LEAKED = leaked
    else:
        maria_again, t_again = traced_answer(WORLD, "maria", PAY_MARIA.question, warm_reader, answers=cache)
        SAFE = (alice_answer.status, leaked, t["served_by"], maria_again.text == maria_answer.text, t_again["served_by"])
        print(f"  and maria, asking again, gets her own answer from the cache: {maria_again.text[:60]}...")
# The re-check before serving: an entry under alice's key that cites a managers-only chunk is never served
probe = AnswerCache(PIPELINE_VERSION, by_cid)
probe.entries[probe.key("alice", PAY_ALICE.question, WORLD.fingerprint)] = maria_answer
RECHECK = probe.get("alice", PAY_ALICE.question, WORLD.fingerprint)
print(f"an entry citing {', '.join(cited_ids(maria_answer))} planted under alice's key: served to alice: "
      f"{RECHECK is not None}")
print()


def longer_passwords(sample: Path) -> None:
    # A policy update: passwords of at least 16 characters instead of 14 (a temporary copy of the documents)
    policy = sample / "acme-handbook" / "password-and-mfa-standard.md"
    policy.write_text(policy.read_text(encoding="utf-8").replace("at least 14 characters", "at least 16 characters"),
                      encoding="utf-8")


updated_chunks = corpus_after(REPO_ROOT, longer_passwords)
updated_index = VectorIndex(index.model, index.revision, index.dimensions)
updated_index.upsert(list(index.records), index.vectors.copy())
plan = sync(updated_index, updated_chunks, embedder.embed)      # Day 05: only the changed chunk is embedded again
UPDATED = World({c.chunk_id: c for c in updated_chunks}, updated_index,
                {u: BM25Index({c.chunk_id: c.text for c in updated_chunks if set(c.allowed_groups) & g})
                 for u, g in DIRECTORY.items()}, fingerprint(snapshot(updated_chunks)))
PASSWORD = by_qid["d07-a10"]                                      # "How long does my password have to be?"
print(f"the password policy changes from 14 to 16 characters: sync embeds {len(plan['add']) + len(plan['re-embed'])} chunk again; index "
      f"fingerprint {WORLD.fingerprint[:16]} -> {UPDATED.fingerprint[:16]}")
STALE = {}
for cache in (NoFingerprintCache(PIPELINE_VERSION, by_cid), AnswerCache(PIPELINE_VERSION, by_cid)):
    before, _ = traced_answer(WORLD, PASSWORD.user, PASSWORD.question, warm_reader, answers=cache)
    after_reader = NamesReader(SentenceCache(embedder), unknown_names_v2, min_score=READER_THRESHOLD)
    after, t = traced_answer(UPDATED, PASSWORD.user, PASSWORD.question, after_reader, answers=cache)
    STALE[type(cache).__name__] = (before.text, after.text, t["served_by"])
    print(f"{type(cache).__name__}: before the update: {before.text}\n  after the update: {after.text} "
          f"(served by {t['served_by']})")

exact caches: 287 questions answered with and without them: identical answers | the first time, 287 answers computed (51 questions embedded; every other question vector, 469 times, came from the query-vector cache); the second time, 287 answers from the cache

QuestionOnlyCache: maria asks, then alice asks the same question -> alice gets [answered] Every March, managers propose pay changes for their team in the compen... (served by answer cache)
  LEAK: it cites manager-pay-review-guide#overview, which only managers may read
AnswerCache: maria asks, then alice asks the same question -> alice gets [abstained] I don't know based on the provided documents.... (served by gate)
  and maria, asking again, gets her own answer from the cache: Every March, managers propose pay changes for their team in ...
an entry citing manager-pay-review-guide#overview planted under alice's key: served to alice: False

the password policy changes from 14 to 16 characters: sync embeds 1 chunk again; index fin

AnswerCache: before the update: Passwords must be at least 14 characters long [password-and-mfa-standard#passwords].
  after the update: Passwords must be at least 16 characters long [password-and-mfa-standard#passwords]. (served by extractive reader (API-free))


The exact caches change no answer. The two broken keys show why the answer cache's key has four parts: **without the asker's groups**, alice is served maria's answer and its managers-only citation (and a permission re-check at serving time would have caught it, which is why it's there too); **without the index fingerprint**, the cache keeps serving "14 characters" after the policy says 16. A cache keyed on content (a sentence vector) can't go stale; one keyed on a name (a question) can.

### 5.3 A semantic cache (D1)

A **semantic cache** serves the answer to an earlier, *similar* question: the nearest cached question (by the cosine of the dense retriever's query vectors, which the pipeline computes anyway) asked by someone with the same groups, under the same index fingerprint, if the cosine is at least **θ**. It saves the whole pipeline, and with a hosted model the model call. It can also serve the right answer to the wrong question.

The pre-registered rule: θ is the lowest value (0.80 to 1.00, in steps of 0.01) at which, leave one out on dev, **no dev question is worse off**. Then the 51 held-out questions are looked up against the dev answers; D1 needs at least one answer served from the cache **and none made worse**.

In [10]:
class SemanticCache:
    """Answers to earlier questions, found by meaning: the nearest earlier question asked by someone with the same
    groups, under the same index fingerprint. It serves that question's answer if the cosine is at least theta, and
    only after checking every cited chunk against the asker's groups (a lookup isn't a search). Answered only."""

    def __init__(self, theta: float, records):
        self.theta, self.records, self.entries = theta, records, []

    def put(self, user: str, index_fingerprint: str, vector, qid: str, a: RagAnswer) -> None:
        if a.status == "answered":                       # a refusal isn't cached
            self.entries.append((tuple(sorted(DIRECTORY[user])), index_fingerprint, vector, qid, a))

    def nearest(self, user: str, index_fingerprint: str, vector, exclude: str | None = None):
        scope = tuple(sorted(DIRECTORY.get(user, ())))
        best_cos, best = -1.0, None
        for groups, fp, v, qid, a in self.entries:
            if groups == scope and fp == index_fingerprint and qid != exclude:
                cos = float(v @ vector)
                if cos > best_cos:
                    best_cos, best = cos, (qid, a)
        return best_cos, best

    def get(self, user: str, index_fingerprint: str, vector, exclude: str | None = None):
        cos, best = self.nearest(user, index_fingerprint, vector, exclude)
        if best is not None and cos >= self.theta and all(may_read(cid, user, self.records) for cid in cited_ids(best[1])):
            return cos, best
        return cos, None


def compare_with_cache(qs, own: dict, nearest: dict, theta: float) -> dict:
    # Each question's outcome when the cache serves its nearest entry (a hit) against the pipeline's own outcome
    hits, better, worse, near, delta = [], [], [], [], {1: 0, 2: 0, 3: 0}
    for q in qs:
        cos, best = nearest[q.id]
        if best is None or cos < theta:
            continue
        hits.append(q.id)
        mine, served = outcome(own[q.id], q), outcome(best[1], q)
        change = better_or_worse(q, served, mine)
        if change and abs(cos - theta) < TIE:               # another CPU's rounding could make this hit a miss
            near.append(q.id)
            continue
        (better if change > 0 else worse if change < 0 else []).append(q.id)
        for c in delta:
            delta[c] += value(served, c) - value(mine, c)
    return {"hits": hits, "better": better, "worse": worse, "near": near, "delta": delta}


# 5.3 The rule, on dev (in-sample): leave one out. Each dev question is looked up against every other dev question's
# answer in its scope; the lowest theta at which no dev question is worse off
dev_cache = SemanticCache(theta=1.0, records=by_cid)
for q in DEV:
    dev_cache.put(q.user, WORLD.fingerprint, warm_queries.embed_query(q.question), q.id, ANSWERS12["names-v2"][q.id])
nearest_dev = {q.id: dev_cache.nearest(q.user, WORLD.fingerprint, warm_queries.embed_query(q.question), exclude=q.id)
               for q in DEV}
own_dev = {q.id: ANSWERS12["names-v2"][q.id] for q in DEV}
dev_curve = {t: compare_with_cache(DEV, own_dev, nearest_dev, t) for t in THETAS}
THETA = min(t for t in THETAS if not dev_curve[t]["worse"])
print(f"Dev, leave one out ({len(DEV)} questions, {len(dev_cache.entries)} answered ones cached; in-sample): highest "
      f"cosine between a dev question and another in its scope: {max(c for c, _ in nearest_dev.values()):.4f}")
print(f"{'theta':>6}{'hits':>6}{'better':>8}{'worse':>7}   value change at cost 1 / 2 / 3")
for t in (0.80, 0.85, 0.88, THETA):
    r = dev_curve[t]
    print(f"{t:>6.2f}{len(r['hits']):>6}{len(r['better']):>8}{len(r['worse']):>7}   "
          + " / ".join(f"{r['delta'][c]:+g}" for c in (1, 2, 3)) + (f"   worse: {', '.join(r['worse'][:5])}"
                                                                    if r["worse"] else ""))
assert THETA == PREREGISTERED["theta"] and {t: (len(dev_curve[t]["hits"]), len(dev_curve[t]["better"]),
                                                len(dev_curve[t]["worse"])) for t in PREREGISTERED["dev curve"]} == \
    PREREGISTERED["dev curve"], "the dev values don't match the pre-registration: stop and find out why"
qid = dev_curve[0.88]["worse"][0]
_, (served_qid, served_answer) = nearest_dev[qid]
print(f"theta = {THETA} (the rule's value) | at 0.88 the one change: {qid} {by_qid[qid].question!r} would be served "
      f"{served_qid}'s answer: {served_answer.text!r} -> {outcome(served_answer, by_qid[qid])}")
print()

# On the 51 v5 test questions (held out): the cache holds the dev answers; each test question is looked up on its own
cache = SemanticCache(THETA, by_cid)
for q in DEV:
    cache.put(q.user, WORLD.fingerprint, warm_queries.embed_query(q.question), q.id, ANSWERS12["names-v2"][q.id])
own_new = {q.id: run(q) for q in NEW}
nearest_new = {q.id: cache.nearest(q.user, WORLD.fingerprint, warm_queries.embed_query(q.question)) for q in NEW}
D1_RESULT = compare_with_cache(NEW, own_new, nearest_new, THETA)
source_of = {q.id: q.note.split(" of ", 1)[1].split(":")[0] for q in NEW}
print(f"v5 test ({len(NEW)} questions; the nearest dev question is the one it was written from for "
      f"{sum(nearest_new[q.id][1] is not None and nearest_new[q.id][1][0] == source_of[q.id] for q in NEW)}): "
      f"served from the cache at theta {THETA}: {len(D1_RESULT['hits'])}")
for q in NEW:
    cos, best = nearest_new[q.id]
    if best is not None and cos >= THETA - 0.05:          # every hit, and every miss within 0.05 of the threshold
        mine, served = outcome(own_new[q.id], q), outcome(best[1], q)
        tag = "HIT " if cos >= THETA else "miss"
        print(f"  {tag} {q.id:<8}{q.tags[0]:<15}{cos:.4f} nearest {best[0]:<8}own {mine:<9}"
              + (f"served {served:<9}" if cos >= THETA else " " * 16) + f"{q.question[:56]}")
by_type = {t: [q for q in NEW if q.tags[0] == t] for t in ("paraphrase", "near-neighbour")}
other_question = [qid for qid in D1_RESULT["hits"] if by_qid[qid].tags[0] == "near-neighbour"]
print(f"near neighbours (written to need another answer) served a dev question's answer: {len(other_question)} "
      f"({', '.join(other_question)}); the pipeline's own answer to them: "
      + ", ".join(f"{qid} {outcome(own_new[qid], by_qid[qid])}" for qid in other_question))
print("hits by type: " + ", ".join(f"{t} {sum(q.id in D1_RESULT['hits'] for q in qs)} of {len(qs)}"
                                  for t, qs in by_type.items()))
for qid in D1_RESULT["better"] + D1_RESULT["worse"]:
    a = own_new[qid]
    gap, what = closest_call(a, READER, rankings["dense"][qid][0][1], GATE, cutoff_gap(by_qid[qid]))
    reader_gap = abs(READER.scored_units(a.context)[0][0] - READER_THRESHOLD) if a.reason.startswith("best sentence") \
        else None
    print(f"{qid}: the pipeline's own answer is {outcome(a, by_qid[qid])} ({a.reason}); its closest call: {what}, "
          f"{gap:.4f}" + (f"; the reader's threshold, which decides it, by {reader_gap:.4f}" if reader_gap else ""))
D1 = len(D1_RESULT["hits"]) >= 1 and not D1_RESULT["worse"]
print(f"better {len(D1_RESULT['better'])}, worse {len(D1_RESULT['worse'])}"
      + (f" ({', '.join(D1_RESULT['worse'])})" if D1_RESULT["worse"] else "")
      + f", near ties set apart {len(D1_RESULT['near'])}, sign test p = "
      f"{sign_test(len(D1_RESULT['better']), len(D1_RESULT['worse'])):.2g} | value change at cost 1 / 2 / 3: "
      + " / ".join(f"{D1_RESULT['delta'][c]:+g}" for c in (1, 2, 3)))
print(f"D1, the semantic cache at theta {THETA}: " + (
    "adopted: it served answers and made none worse" if D1 else
    "not adopted: " + ("it served no answer from the cache" if not D1_RESULT["hits"] else
                       f"it made {len(D1_RESULT['worse'])} outcome(s) worse") + "; answers are cached exactly only"))
print()
test_curve = {t: compare_with_cache(NEW, own_new, nearest_new, t) for t in THETAS}
print("as a description (never used to decide): every threshold on the test questions")
print(f"{'theta':>6}" + "".join(f"{t:>6.2f}" for t in THETAS[::2]))
for label, key in (("hits", "hits"), ("better", "better"), ("worse", "worse")):
    print(f"{label:>6}" + "".join(f"{len(test_curve[t][key]):>6}" for t in THETAS[::2]))

Dev, leave one out (236 questions, 180 answered ones cached; in-sample): highest cosine between a dev question and another in its scope: 0.8890
 theta  hits  better  worse   value change at cost 1 / 2 / 3
  0.80    53       2     14   -24 / -38 / -52   worse: d03-01, d03-07, d03-08, d01-03, d07-a01
  0.85    16       0      4   -8 / -12 / -16   worse: d07-a06, d11-a32, d12-a16, d12-a25
  0.88     6       0      1   -2 / -3 / -4   worse: d12-a25
  0.89     0       0      0   +0 / +0 / +0
theta = 0.89 (the rule's value) | at 0.88 the one change: d12-a25 'Does being the primary or the secondary caregiver change how much parental leave I get?' would be served d12-a24's answer: 'Secondary caregivers receive 6 weeks of fully paid leave [parental-leave-policy#leave-entitlement].' -> wrong

v5 test (51 questions; the nearest dev question is the one it was written from for 34): served from the cache at theta 0.89: 10
  miss d13-a01 paraphrase     0.8616 nearest d03-01  own correct              

**Not adopted.** On dev the rule gives θ = 0.89, a value no pair of dev questions reaches (dev has no paraphrases, so it shows only what a cache gets wrong). On the held-out questions the cache served 10 answers, and one was worse: d13-u13 asks whether Acme matches charity donations, which the documents don't cover; the pipeline refuses it, but its dev twin (d10-u05) was answered, wrongly, and the cache spread that wrong answer to the paraphrase (the pipeline's own refusal isn't a near tie: the reader's threshold decides it by 0.006). Five of the ten hits were **near neighbours**, written to need another answer: they were served another question's answer, and the outcome counts call that "no change" only because the pipeline gets those four wrong too (the fifth, "Who approves a browser extension?", was served the right answer only because a dev question had asked nearly the same thing). Lower thresholds serve more and get more wrong; from 0.95 up, nothing is served. On these documents, a similarity high enough to be safe is too high to be useful. Answers are cached **exactly** only.

## 6. Timeouts, retries, a circuit breaker and a fallback

Day 06's client sends one request and waits up to 60 seconds for each read. Today's `ResilientChatModel` is the same client (the same request, the same key handling) with five additions:

| | What it does | Why |
|---|---|---|
| **Timeout per attempt** | bounds each blocking step: the connection and each read | a stalled server can't hold a request forever |
| **Deadline** | bounds the whole model step, retries and waits included, checked while the reply arrives | a timeout per read isn't a deadline: a server that sends a few bytes at a time never trips it |
| **Retries** | up to 3 attempts, only for failures that can pass (connection errors, timeouts, 408, 409, 429, 5xx), never for a key or billing problem; `Retry-After` honoured if it fits in the deadline | as OpenAI's client and error-code guide recommend |
| **Backoff with jitter** | a random wait between 0 and min(8 s, 0.5 s × 2^retry) | spreads retries out, so clients that failed together don't retry together |
| **Circuit breaker** | after 3 failed requests in a row, stops calling the model for 30 s, then lets one trial request through | during an outage, answers at once instead of waiting for timeouts on every request |

When the model can't answer, the **fallback** answers (here, the API-free reader on the same context, checked by the same citation check); with no fallback, the pipeline shows Day 06's "unavailable" message. Every attempt, wait and fallback is a span in the trace. A local fake server misbehaves on purpose:

In [11]:
RETRY_STATUSES = {408, 409, 429} | set(range(500, 600))       # as the OpenAI Python library retries
BILLING_CODES = {"insufficient_quota", "credit_balance_exhausted", "organization_spend_limit_exceeded",
                 "project_spend_limit_exceeded", "organization_usage_limit_exceeded"}   # a 429 retrying can't fix


class CallFailed(GeneratorError):
    """One failed attempt, with what the retry logic needs: its kind, whether it can pass, and how long the server asked
    us to wait. Built from the status code and the error code only: a server's message may echo the key."""

    def __init__(self, kind: str, message: str, retryable: bool, retry_after: float | None = None):
        super().__init__(message)
        self.kind, self.retryable, self.retry_after = kind, retryable, retry_after


def retry_after_seconds(headers) -> float | None:
    for header, divisor in (("retry-after-ms", 1000), ("retry-after", 1)):   # (an HTTP date is allowed too; not here)
        value = headers.get(header) if headers else None
        if value is not None:
            try:
                return float(value) / divisor
            except ValueError:
                continue
    return None


class CircuitBreaker:
    """After `failures` failed requests in a row, stop calling the model for `cooldown` seconds (every request goes
    straight to the fallback); then let one trial request through: if it works, close again."""

    def __init__(self, failures: int, cooldown: float, clock=time.monotonic):
        self.failures, self.cooldown, self.clock = failures, cooldown, clock
        self.failed_in_a_row, self.opened_at = 0, None

    @property
    def state(self) -> str:
        if self.opened_at is None:
            return "closed"
        return "open" if self.clock() - self.opened_at < self.cooldown else "half-open"

    def allow(self) -> bool:
        return self.state != "open"

    def record(self, ok: bool) -> None:
        if ok:
            self.failed_in_a_row, self.opened_at = 0, None
        else:
            trial = self.state == "half-open"
            self.failed_in_a_row += 1
            if trial or self.failed_in_a_row >= self.failures:
                self.opened_at = self.clock()


class ResilientChatModel(ChatModel):
    """Day 06's ChatModel (same request, same key handling), with a timeout per attempt, a deadline for the whole step,
    retries with backoff and jitter for failures that can pass, an optional circuit breaker, and a fallback generator.
    Every attempt, wait and fallback is a span in the request's trace."""
    traced = True                                       # traced_answer() passes it the request's trace

    def __init__(self, provider, model, api_key=None, base_url=None, *, fallback=None, attempts=3, timeout=10.0,
                 deadline=30.0, backoff_base=0.5, backoff_cap=8.0, breaker=None, seed=None, max_output_tokens=1000):
        super().__init__(provider, model, api_key, base_url, max_output_tokens=max_output_tokens, timeout=timeout)
        self.fallback, self.attempts, self.deadline, self.breaker = fallback, attempts, deadline, breaker
        self.backoff_base, self.backoff_cap, self.rng = backoff_base, backoff_cap, random.Random(seed)
        self.calls = 0                                  # HTTP requests sent, retries included

    def attempt(self, context, deadline: float) -> Reply:
        # One HTTP request. The socket timeout bounds each blocking step (connect, each read); the deadline is checked
        # while the reply arrives, so a server that sends a few bytes at a time can't hold the request open.
        headers = {"Content-Type": "application/json"}
        if self._api_key:
            headers["Authorization"] = f"Bearer {self._api_key}"
        body = json.dumps(self.request_body(context)).encode("utf-8")
        local = urllib.parse.urlsplit(self.url).hostname in LOCAL_HOSTS
        opener = urllib.request.build_opener(*([urllib.request.ProxyHandler({})] if local else []))
        self.calls += 1
        try:
            with opener.open(urllib.request.Request(self.url, body, headers),
                             timeout=max(0.001, min(self.timeout, deadline - time.monotonic()))) as response:
                pieces = []
                while piece := response.read1(65536):
                    pieces.append(piece)
                    if time.monotonic() > deadline:
                        raise CallFailed("deadline", "the reply was still arriving at the deadline", False)
            data = json.loads(b"".join(pieces))
        except CallFailed:
            raise
        except urllib.error.HTTPError as error:         # the server answered with an error status
            try:
                code = (json.loads(error.read().decode("utf-8", "replace")).get("error") or {}).get("code")
            except Exception:
                code = None
            retryable = error.code in RETRY_STATUSES and code not in BILLING_CODES and \
                (error.headers or {}).get("x-should-retry") != "false"
            message = f"HTTP {error.code}" + (", authentication failed: check RAG_CHAT_API_KEY in .env"
                                              if error.code in (401, 403) else f" ({code})" if code else "")
            raise CallFailed(f"HTTP {error.code}", message, retryable, retry_after_seconds(error.headers)) from None
        except TimeoutError:
            raise CallFailed("timeout", f"no reply within {self.timeout:g} s", True) from None
        except urllib.error.URLError as error:
            if isinstance(error.reason, TimeoutError):
                raise CallFailed("timeout", f"no connection within {self.timeout:g} s", True) from None
            raise CallFailed("unreachable", f"could not reach {self.url}", True) from None
        except OSError:                                 # the connection dropped half-way
            raise CallFailed("connection lost", "the connection was lost", True) from None
        except ValueError:                              # a proxy's error page, a wrong URL, ...
            raise CallFailed("not JSON", "the reply was not JSON", False) from None
        try:
            choice = data["choices"][0]
            text = choice["message"].get("content") or ""
        except (KeyError, IndexError, TypeError, AttributeError):
            raise CallFailed("bad format", "unexpected response format", False) from None
        if not text.strip():
            raise CallFailed("empty reply", f"empty reply (finish_reason: {choice.get('finish_reason')})", False)
        usage = data.get("usage") or {}
        return Reply(text.strip(), self.name, usage={"input_tokens": usage.get("prompt_tokens"),
                                                     "output_tokens": usage.get("completion_tokens")})

    def generate(self, context, trace=None) -> Reply:
        span = trace.span if trace is not None else (lambda name, **attributes: nullcontext({}))
        deadline = time.monotonic() + self.deadline
        if self.breaker is not None and not self.breaker.allow():
            return self.fall_back(context, "circuit open", "the circuit breaker is open", span)
        failure = None
        for number in range(1, self.attempts + 1):
            with span(f"chat {self.model}", parent="generate", attempt=number, **{"gen_ai.operation.name": "chat",
                                                               "gen_ai.provider.name": self.provider,
                                                               "gen_ai.request.model": self.model}) as s:
                try:
                    reply = self.attempt(context, deadline)
                    s.update({"outcome": "ok", "gen_ai.usage.input_tokens": reply.usage.get("input_tokens"),
                              "gen_ai.usage.output_tokens": reply.usage.get("output_tokens")})
                except CallFailed as error:
                    reply, failure = None, error
                    s.update({"outcome": error.kind, "error.type": error.kind, "retryable": error.retryable})
            if reply is not None:
                if self.breaker is not None:
                    self.breaker.record(True)
                return reply
            if not failure.retryable or number == self.attempts:
                break
            wait = failure.retry_after if failure.retry_after is not None else \
                self.rng.uniform(0, min(self.backoff_cap, self.backoff_base * 2 ** (number - 1)))   # full jitter
            if wait >= deadline - time.monotonic():     # no time left for another attempt: stop now
                break
            with span("backoff", parent="generate", wait_ms=round(wait * 1000, 1),
                      reason="Retry-After" if failure.retry_after is not None else "jitter"):
                time.sleep(wait)
        if self.breaker is not None:
            self.breaker.record(False)
        return self.fall_back(context, failure.kind, str(failure), span)

    def fall_back(self, context, cause: str, message: str, span) -> Reply:
        if self.fallback is None:
            raise GeneratorError(f"{self.name}: {message}")     # Day 06's pipeline shows "unavailable" for this
        with span("fallback", parent="generate", cause=cause, generator=self.fallback.name):
            reply = self.fallback.generate(context)
        return Reply(reply.text, f"{self.fallback.name}, fallback after: {cause}", reply.note, reply.usage)


# ----------------------------------------------------------------------------- a local server that misbehaves on purpose
class QuietServer(ThreadingHTTPServer):
    daemon_threads = True

    def handle_error(self, request, client_address):
        # A client that gave up first (a timeout, a deadline) closes the connection while the server still writes:
        # expected here, so not printed. Anything else still is.
        if not isinstance(sys.exc_info()[1], (ConnectionAbortedError, ConnectionResetError, BrokenPipeError)):
            super().handle_error(request, client_address)


class FakeModel(BaseHTTPRequestHandler):
    """A stand-in for an OpenAI-compatible server. Each request takes the next step of `script`: ("ok",),
    ("status", code, headers, error_code), ("slow", seconds), ("trickle", seconds), ("not json",)."""
    script, requests = [], 0

    def do_POST(self):
        body = json.loads(self.rfile.read(int(self.headers["Content-Length"])))
        FakeModel.requests += 1
        step = FakeModel.script.pop(0) if FakeModel.script else ("ok",)
        source = re.search(r'<source id="([^"]+)"', body["messages"][1]["content"])
        reply = f"This is the fake model's answer [{source.group(1) if source else 'none'}]."
        ok = json.dumps({"choices": [{"finish_reason": "stop", "message": {"content": reply}}],
                         "usage": {"prompt_tokens": 400, "completion_tokens": 12}}).encode("utf-8")
        if step[0] == "slow":
            time.sleep(step[1])
        if step[0] == "status":
            status, headers, code = step[1], step[2], step[3]
            data = json.dumps({"error": {"message": f"denied for {self.headers.get('Authorization')}",
                                         "code": code}}).encode("utf-8")          # it echoes the key, on purpose
        else:
            status, headers, data = 200, {}, (b"<html>bad gateway</html>" if step[0] == "not json" else ok)
        self.send_response(status)
        for name, header_value in headers.items():
            self.send_header(name, header_value)
        self.send_header("Content-Length", str(len(data)))
        self.end_headers()
        if step[0] == "trickle":                        # a few bytes at a time: no single read waits long
            for i in range(0, len(data), 8):
                self.wfile.write(data[i:i + 8])
                self.wfile.flush()
                time.sleep(step[1] / (len(data) / 8))
        else:
            self.wfile.write(data)

    def log_message(self, *args):
        pass


server = QuietServer(("127.0.0.1", 0), FakeModel)
threading.Thread(target=server.serve_forever, daemon=True).start()
FAKE_URL = f"http://127.0.0.1:{server.server_address[1]}/v1"
FAKE_KEY = "fake-key-for-the-day-13-demo"
print(f"a fake model server on {FAKE_URL} (local; no request leaves this machine) | client settings, 20x faster than "
      f"real use: {DEMO['attempts']} attempts, {DEMO['timeout']} s per attempt, {DEMO['deadline']} s deadline, "
      f"backoff up to {DEMO['backoff_cap']} s")

a fake model server on http://127.0.0.1:39367/v1 (local; no request leaves this machine) | client settings, 20x faster than real use: 3 attempts, 0.5 s per attempt, 1.5 s deadline, backoff up to 0.4 s


In [12]:
def demo_model(**changes) -> ResilientChatModel:
    settings = {**DEMO, "fallback": READER, "seed": 13, **changes}
    return ResilientChatModel("openai", "gpt-6-luna", FAKE_KEY, FAKE_URL, **settings)


SCENARIOS = {   # what the server does, request by request
    "healthy": [("ok",)],
    "one 503, then fine": [("status", 503, {}, "server_is_overloaded"), ("ok",)],
    "429 with Retry-After 0.2 s": [("status", 429, {"Retry-After": "0.2"}, "rate_limit_exceeded"), ("ok",)],
    "429 with Retry-After 5 s": [("status", 429, {"Retry-After": "5"}, "rate_limit_exceeded"), ("ok",)],
    "429, no credit left": [("status", 429, {}, "credit_balance_exhausted"), ("ok",)],
    "401, wrong key": [("status", 401, {}, "invalid_api_key"), ("ok",)],
    "500 three times": [("status", 500, {}, None)] * 3,
    "slow (2 s per reply)": [("slow", 2.0)] * 3,
    "trickle (2.5 s per reply)": [("trickle", 2.5)] * 3,
    "not JSON (a proxy page)": [("not json",), ("ok",)],
}
q_demo = LOST                                           # any question that reaches the generator
RESILIENCE = {}
print(f"{'server':<28}{'HTTP calls':>11}{'waited':>9}{'took':>8}   {'answered by':<17}attempts")
for name, script in SCENARIOS.items():
    FakeModel.script, FakeModel.requests = list(script), 0
    model = demo_model()
    started = time.perf_counter()
    a, t = traced_answer(WORLD, q_demo.user, q_demo.question, model, request_id=f"demo-{len(RESILIENCE) + 1:02d}")
    took = time.perf_counter() - started
    waited = sum(s["wait_ms"] for s in t["spans"] if s["name"] == "backoff") / 1000
    attempts = [s["outcome"] for s in t["spans"] if s["name"].startswith("chat ")]
    RESILIENCE[name] = {"calls": model.calls, "attempts": attempts, "waited": waited, "took": took,
                        "served_by": t["served_by"], "status": a.status, "trace": t}
    print(f"{name:<28}{model.calls:>11}{waited:>8.2f}s{took:>7.2f}s   "
          f"{'the model' if t['served_by'] == model.name else 'fallback reader':<17}{', '.join(attempts)}")
print()
show_trace(RESILIENCE["one 503, then fine"]["trace"])
print()
# Without a fallback, a failure is Day 06's "unavailable", and the key the server echoed never reaches the user
FakeModel.script = [("status", 401, {}, "invalid_api_key")]
try:
    demo_model(fallback=None).generate(run(q_demo).context)
except GeneratorError as error:
    NO_FALLBACK = str(error)
print(f"no fallback, 401: GeneratorError({NO_FALLBACK!r}) -> the pipeline shows Day 06's 'unavailable' message")

# A timeout isn't a deadline: Day 06's client waits for every byte of a trickling reply, each one inside its timeout
FakeModel.script = [("trickle", 2.5)]
started = time.perf_counter()
day06_reply = ChatModel("openai", "gpt-6-luna", FAKE_KEY, FAKE_URL, timeout=DEMO["timeout"]).generate(run(q_demo).context)
DAY06_TRICKLE = time.perf_counter() - started
print(f"Day 06's ChatModel with a {DEMO['timeout']} s timeout, same trickling server: waited {DAY06_TRICKLE:.2f} s and "
      f"returned {day06_reply.text[:36]!r}...; the resilient client gave up at its {DEMO['deadline']} s deadline "
      f"({RESILIENCE['trickle (2.5 s per reply)']['took']:.2f} s)")

print()
# An outage: every request fails. Retries multiply the calls to a server that is already struggling; a circuit breaker
# stops calling it after a few failed requests and answers from the fallback at once, then tries again after a while.
def outage(model, requests: int = 8) -> tuple[int, float, Counter]:
    FakeModel.script, FakeModel.requests = [("status", 503, {}, "server_is_overloaded")] * 100, 0
    started, served = time.perf_counter(), Counter()
    for i in range(requests):
        a, t = traced_answer(WORLD, q_demo.user, q_demo.question, model, request_id=f"outage-{i + 1:02d}")
        served["the model" if t["served_by"] == model.name else "fallback"] += 1
    return FakeModel.requests, time.perf_counter() - started, served


breaker = CircuitBreaker(**DEMO_BREAKER)
OUTAGE = {"retries only": outage(demo_model()), "retries and a breaker": outage(demo_model(breaker=breaker))}
for name, (calls, took, served) in OUTAGE.items():
    print(f"an outage, 8 requests, {name:<22}: {calls:>2} calls to the failing server, {took:.2f} s, answered by "
          + ", ".join(f"{k} {n}" for k, n in served.items()))
state_during = breaker.state
time.sleep(DEMO_BREAKER["cooldown"])                      # the server recovers; after the cool-down, one trial request
FakeModel.script, FakeModel.requests = [("ok",)], 0
a, t = traced_answer(WORLD, q_demo.user, q_demo.question, demo_model(breaker=breaker), request_id="after-outage")
BREAKER_AFTER = (state_during, FakeModel.requests, t["served_by"], breaker.state)
print(f"breaker during the outage: {state_during}; {DEMO_BREAKER['cooldown']} s later one trial request "
      f"({FakeModel.requests} call) is answered by {t['served_by']} -> breaker {breaker.state}")

server                       HTTP calls   waited    took   answered by      attempts
healthy                               1    0.00s   0.03s   the model        ok
one 503, then fine                    2    0.01s   0.03s   the model        HTTP 503, ok


429 with Retry-After 0.2 s            2    0.20s   0.27s   the model        HTTP 429, ok
429 with Retry-After 5 s              1    0.00s   0.02s   fallback reader  HTTP 429
429, no credit left                   1    0.00s   0.02s   fallback reader  HTTP 429
401, wrong key                        1    0.00s   0.02s   fallback reader  HTTP 401
500 three times                       3    0.04s   0.08s   fallback reader  HTTP 500, HTTP 500, HTTP 500


slow (2 s per reply)                  3    0.04s   1.52s   fallback reader  timeout, timeout, timeout


trickle (2.5 s per reply)             1    0.00s   1.55s   fallback reader  deadline
not JSON (a proxy page)               1    0.00s   0.02s   fallback reader  not JSON

demo-02  user alice ['all-employees']  question sha256 e5a84cae394b1030 (8 words)  index 3f8344d00b8b4878  pipeline day-13.1
  embed_query           0.0    15.6 ms ###########              cache_hit False
  retrieve.bm25        15.6     0.1 ms #                        candidates 11; top travel-policy#safety-while-travelling 7.2761 (+2)
  retrieve.dense       15.7     0.2 ms #                        candidates 43; top security-incident-reporting#what-not-to-do 0.6758 (+2)
  retrieve.fuse        15.9     0.1 ms #                        gen_ai.operation.name retrieval; gen_ai.retrieval.top_k 3; documents travel-policy#safety-while-travelling 0.9485 (+2); gap_after_last 0.0453
  gate                 15.9     0.0 ms #                        threshold 0.62; dense_top 0.6758; decision pass
  context              16.0     0.1

Day 06's ChatModel with a 0.5 s timeout, same trickling server: waited 2.44 s and returned "This is the fake model's answer [tra"...; the resilient client gave up at its 1.5 s deadline (1.55 s)



an outage, 8 requests, retries only          : 24 calls to the failing server, 0.55 s, answered by fallback 8
an outage, 8 requests, retries and a breaker :  9 calls to the failing server, 0.34 s, answered by fallback 8


breaker during the outage: open; 1.5 s later one trial request (1 call) is answered by gpt-6-luna via openai -> breaker closed


Each failure gets the treatment it needs: a 503 or a rate limit is retried (after exactly the `Retry-After` the server asked for), a `Retry-After` longer than the deadline isn't waited for, an exhausted balance or a wrong key isn't retried at all (retrying can't fix them), a slow server costs three timeouts and a trickling one costs the deadline, where Day 06's client waited for the whole trickle. In the outage, retries alone send 24 calls to a server that is already failing; with the breaker, 9, and the requests after the third go straight to the fallback until the trial request finds the server healthy again. The key the fake server echoes never reaches a trace or an error message.

## 7. What to answer when the model fails (D2)

When the model can't answer, the pipeline can still answer with the API-free reader, or show "unavailable". Under Day 12's value of an answer (correct +1, refused 0, wrong −2), "unavailable" is worth 0 for every question, so the reader is worth serving only if its total value is above 0.

In [13]:
# D2: is the API-free reader worth serving when the model fails? Refusing everything ("unavailable") is worth 0 under
# Day 12's value of an answer, so the fallback is worth having if the current pipeline's total value is above 0.
dev_counts = counts(GRADES12["names-v2"], DEV)
dev_values = {c: counts(GRADES12["names-v2"], DEV, c)[5] for c in (1, 2, 3)}
assert dev_counts[:5] == PREREGISTERED["dev outcomes"] and dev_values == PREREGISTERED["dev value"], \
    "the dev values don't match the pre-registration: stop and find out why"
GRADES_NEW = {q.id: grade(own_new[q.id], q, sent_of(q)) for q in NEW}


def outcome_table(grades: dict, qs, label: str) -> None:
    answerable, unanswerable = [q for q in qs if q.answerable], [q for q in qs if not q.answerable]
    c = counts(grades, qs)
    print(f"{label} ({len(answerable)} answerable, {len(unanswerable)} not):")
    for name, n, total in (("correct", c[0], len(answerable)), ("wrong", c[1], len(answerable)),
                           ("refused", c[2], len(answerable)), ("unanswerable refused", c[3], len(unanswerable)),
                           ("unanswerable answered", c[4], len(unanswerable))):
        low, high = wilson(n, total)
        print(f"  {name:<24}{n:>4} of {total:<4}{n / total:>5.0%} ({low:.0%} to {high:.0%})")
    stages = Counter(grades[q.id].stage for q in qs if grades[q.id].stage != "ok")
    print("  not ok: " + ", ".join(f"{s} {n}" for s, n in stages.most_common()))


print(f"Dev (in-sample, {len(DEV)} questions): answerable {dev_counts[0]} correct / {dev_counts[1]} wrong / "
      f"{dev_counts[2]} refused, unanswerable {dev_counts[3]} refused / {dev_counts[4]} answered | value at cost "
      "1 / 2 / 3: " + " / ".join(f"{dev_values[c]:+g}" for c in (1, 2, 3)))
outcome_table(GRADES_NEW, NEW, "v5 test, the current pipeline (what the fallback would answer)")
D2_VALUE = {c: sum(value(GRADES_NEW[q.id].outcome, c) for q in NEW) for c in (1, 2, 3)}
D2 = D2_VALUE[COST] > 0
print(f"value of answering with it instead of 'unavailable' (worth 0), at cost 1 / 2 / 3: "
      + " / ".join(f"{D2_VALUE[c]:+g}" for c in (1, 2, 3)))
near_d2 = {q.id: closest_call(own_new[q.id], READER, rankings["dense"][q.id][0][1], GATE, cutoff_gap(q)) for q in NEW}
near_d2 = {qid: call for qid, call in near_d2.items() if call[0] < TIE}
# The most a near tie could add if its close call went the other way: a wrong answer becoming correct (+3 at cost 2), a
# refusal becoming correct (+1), an answered unanswerable question becoming a refusal (+2); a correct answer can't gain
best_gain = {qid: {"wrong": 1 + COST, "refused": 1 if by_qid[qid].answerable else 0, "answered": COST,
                   "correct": 0}[GRADES_NEW[qid].outcome] for qid in near_d2}
print(f"answers decided by a gap under {TIE}: {len(near_d2)} (" + ", ".join(
    f"{qid} {GRADES_NEW[qid].outcome}, {what}" for qid, (gap, what) in near_d2.items()) + f"); even if every one of "
      f"them went the best way, the value would be {D2_VALUE[COST] + sum(best_gain.values()):+g} at cost {COST}")
print(f"D2, the API-free reader as the fallback when the model fails: " + (
    f"adopted (value {D2_VALUE[COST]:+g} at cost {COST}): later days answer with it, marked as the fallback's answer"
    if D2 else f"not adopted (value {D2_VALUE[COST]:+g} at cost {COST}): later days show the 'unavailable' message"))
print()
# As a description: does the pipeline give a paraphrase the same outcome as the question it was written from?
pairs = [(q, by_qid[source_of[q.id]]) for q in NEW if q.tags[0] == "paraphrase"]
same = [q.id for q, src in pairs if GRADES_NEW[q.id].outcome == GRADES12["names-v2"][src.id].outcome]
print(f"paraphrases with the same outcome as the dev question they reword: {len(same)} of {len(pairs)}; different: "
      + ", ".join(f"{q.id} {GRADES12['names-v2'][src.id].outcome}->{GRADES_NEW[q.id].outcome}"
                  for q, src in pairs if q.id not in same))

Dev (in-sample, 236 questions): answerable 124 correct / 23 wrong / 17 refused, unanswerable 39 refused / 33 answered | value at cost 1 / 2 / 3: +68 / +12 / -44
v5 test, the current pipeline (what the fallback would answer) (33 answerable, 18 not):
  correct                   20 of 33    61% (44% to 75%)
  wrong                      7 of 33    21% (11% to 38%)
  refused                    6 of 33    18% (9% to 34%)
  unanswerable refused       6 of 18    33% (16% to 56%)
  unanswerable answered     12 of 18    67% (44% to 84%)
  not ok: answered: nothing answers it 12, gate 5, reader: answer in the context, not used 3, reader: right source, wrong part 3, names rule: a name 1, retrieval: answer not in the context 1
value of answering with it instead of 'unavailable' (worth 0), at cost 1 / 2 / 3: +1 / -18 / -37
answers decided by a gap under 0.001: 7 (d13-a05 correct, which sections are sent, d13-a06 wrong, which source is copied, d13-a08 wrong, which sentences are copied, d13-a11 refuse

**Not adopted.** In-sample on dev the reader is worth +12 at cost 2; on the held-out questions, −18: it answers 12 of the 18 unanswerable ones. The held-out set leans that way by construction (its unanswerable questions are near neighbours of answerable ones, written to look answerable), so this says less about traffic than about the reader: **it can't tell a near neighbour from the question it resembles**, and Day 12's live pass showed a model doing much better at that (it refused all 10 unanswerable questions it was sent). Later days show "unavailable" when the model fails. The paraphrases make the same point another way: a third of them get a different outcome from the question they reword.

## 8. New regression cases

Every failure a lesson records becomes a case in [`evals/regressions.jsonl`](../evals/regressions.jsonl), run by `tests/test_regressions.py` from `src/` only. Day 12's live pass recorded three (its replies are in [`evals/reviews/day-12-live-gpt-6-luna.jsonl`](../evals/reviews/day-12-live-gpt-6-luna.jsonl), with the author's reading of each), and section 6 one:

In [14]:
# Failures recorded since Day 12, now regression cases (evals/regressions.jsonl; tests/test_regressions.py runs them
# from src/ only). Three come from Day 12's live pass (evals/reviews/day-12-live-gpt-6-luna.jsonl), one from section 6.
live_reply = {r["id"]: r["model_reply"] for r in live12}
NEW_CASES = {}
for qid, why in (("d12-a05", "the question's own number"), ("d12-a20", "a one-word 'Yes.'")):
    a = run(by_qid[qid], FixedReply(live_reply[qid]))
    NEW_CASES[qid] = (a.status, a.check.problems)
    print(f"{qid}: gpt-6-luna's correct reply {live_reply[qid]!r}\n  -> {a.status}: {'; '.join(a.check.problems)} "
          f"({why}: the citation check fails a correct reply closed)")
a = run(LOST, FixedReply(live_reply[LOST.id]))
g = grade(a, LOST, sent_of(LOST))
NEW_CASES[LOST.id] = (a.status, g.outcome, g.label_in_context, g.cites_label)
print(f"{LOST.id}: gpt-6-luna's reply {live_reply[LOST.id]!r}\n  -> {a.status}, {g.outcome}: {g.stage} (the model, like "
      f"the reader, used the travel policy's sentence; 'within one hour' was in its context)")
print(f"Day 06's ChatModel and a trickling server: {DAY06_TRICKLE:.2f} s with a {DEMO['timeout']} s timeout (a timeout "
      "per read isn't a deadline)")
added = [r for r in regressions if r["check"] in ("citation_check_false_block", "model_ignores_answer",
                                                 "timeout_is_not_a_deadline")]
print("in evals/regressions.jsonl: " + ", ".join(f"{r['id']} {r['status']} ({r['check']})" for r in added))

d12-a05: gpt-6-luna's correct reply 'No, the policy requires an itemised receipt for expenses over USD 25, so a USD 12 coffee does not require one. [expense-reimbursement-policy#submitting-a-claim]'
  -> blocked: 12 is not in the cited source (the question's own number: the citation check fails a correct reply closed)
d12-a20: gpt-6-luna's correct reply 'Yes. You can contact the IT Service Desk through the #it-help chat channel. [new-starter-it-checklist#getting-help]'
  -> blocked: no citation (a one-word 'Yes.': the citation check fails a correct reply closed)
d03-13: gpt-6-luna's reply 'Report a lost laptop straight away. [travel-policy#safety-while-travelling]'
  -> answered, wrong: reader: answer in the context, not used (the model, like the reader, used the travel policy's sentence; 'within one hour' was in its context)
Day 06's ChatModel and a trickling server: 2.44 s with a 0.5 s timeout (a timeout per read isn't a deadline)
in evals/regressions.jsonl: R30 known (citation_check

All four are **known** failures, pinned as they behave now, so a fix shows up as a failing test that someone has to look at. The two citation-check cases are the cost of failing closed: a check strict enough to stop a changed number also stops a number the question itself contains, and a one-word "Yes.". Fixing them belongs to a day that changes `answering.py`, measured like any other change.

## 9. Optional: a live pass through the resilient client

Off by default. With `RAG_CHAT_*` in `.env` and `DAY13_LIVE_MODEL=1`, 20 questions Day 12's live pass asked go through the resilient client (the real settings above, the breaker, the API-free reader as the fallback), traced. Each trace is checked for text and the key before it is written; the replies go to `outputs/day-13/live-run.jsonl`, the traces to `live-traces.jsonl` (both git-ignored). It reports each request's steps, the model call's p50 and p95, tokens and cost, and how many replies are identical to the ones the same model gave on 2026-11-12. One run of one model: an observation, not a measurement.

In [15]:
LIVE_MODEL = os.environ.get("DAY13_LIVE_MODEL") == "1"     # or set it to True here (a hosted model costs money)
MAX_LIVE_REQUESTS = 20
answerable12 = [r for r in live12[6:] if by_qid[r["id"]].answerable]
unanswerable12 = [r for r in live12 if not by_qid[r["id"]].answerable]
LIVE_ROWS = live12[:6] + answerable12[:9] + unanswerable12[:5]      # questions Day 12's live pass asked, in its order
assert len(LIVE_ROWS) <= MAX_LIVE_REQUESTS and len({r["id"] for r in LIVE_ROWS}) == len(LIVE_ROWS)
longest = max(sum(tokens_estimate(m["content"]) for m in build_messages(run(by_qid[r["id"]]).context)) for r in LIVE_ROWS)
price_in, price_out = PRICES["gpt-6-luna"]
print(f"live pass: {len(LIVE_ROWS)} questions Day 12's live pass asked (its {len(live12[:6])} recorded failures, then "
      f"{len(answerable12[:9])} answerable and {len(unanswerable12[:5])} unanswerable ones); gpt-6-luna at most USD "
      f"{len(LIVE_ROWS) * (longest * price_in + 1000 * price_out) / 1e6:.4f} (USD {price_in} / {price_out} per million "
      "tokens, checked 2026-11-12) | client: " + ", ".join(f"{k} {v:g}" for k, v in CLIENT.items())
      + f", breaker after {BREAKER['failures']} failed requests for {BREAKER['cooldown']:g} s, fallback: the API-free reader")
LIVE = {}
if not LIVE_MODEL:
    print("Live model: off. Fill in RAG_CHAT_* in .env (see .env.example), then set LIVE_MODEL = True.")
else:
    env = read_env_file(REPO_ROOT / ".env")
    if not env.get("RAG_CHAT_PROVIDER") or not env.get("RAG_CHAT_MODEL"):
        raise ValueError("Set RAG_CHAT_PROVIDER and RAG_CHAT_MODEL in .env (see .env.example)")
    live = ResilientChatModel(env["RAG_CHAT_PROVIDER"], env["RAG_CHAT_MODEL"], env.get("RAG_CHAT_API_KEY") or None,
                              env.get("RAG_CHAT_BASE_URL") or None, fallback=READER,
                              breaker=CircuitBreaker(**BREAKER), seed=13, **CLIENT)
    secrets = [env.get("RAG_CHAT_API_KEY") or ""]
    print(f"Live model: {live.name} at {live.url}\n")
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    with (OUT_DIR / "live-run.jsonl").open("w", encoding="utf-8") as log, \
            (OUT_DIR / "live-traces.jsonl").open("w", encoding="utf-8") as trace_log:
        for i, r in enumerate(LIVE_ROWS, start=1):
            q = by_qid[r["id"]]
            a, t = traced_answer(WORLD, q.user, q.question, live, request_id=f"live-{i:02d}")
            assert not leaks(t, q.question, secrets), f"{q.id}: the trace holds text or a key"
            g = grade(a, q, sent_of(q))
            text = a.reply.text if a.reply else None
            LIVE[q.id] = (a, g, t, text == r["model_reply"])
            print(f"{q.id} [{a.status}] {g.outcome}: {g.stage} | served by {t['served_by']} | Day 12: "
                  f"{r['rubric']['outcome']}, same reply: {text == r['model_reply']}\n  {q.question}\n  {text}")
            log.write(json.dumps({"id": q.id, "question": q.question, "status": a.status, "served_by": t["served_by"],
                                  "model_reply": text, "grade": dataclasses.asdict(g), "day12_same_reply":
                                  text == r["model_reply"], "usage": a.reply.usage if a.reply else {},
                                  "ms": t["ms"]}, ensure_ascii=False) + "\n")
            trace_log.write(json.dumps(t, ensure_ascii=False) + "\n")
    calls = [s for a, g, t, same in LIVE.values() for s in t["spans"] if s["name"].startswith("chat ")]
    ok = [s["ms"] for s in calls if s["outcome"] == "ok"]
    used_in = sum(s.get("gen_ai.usage.input_tokens") or 0 for s in calls)
    used_out = sum(s.get("gen_ai.usage.output_tokens") or 0 for s in calls)
    print(f"\n{len(LIVE)} requests, {len(calls)} calls ({len(calls) - len(LIVE)} retried), fallbacks "
          f"{sum('fallback' in t['served_by'] for a, g, t, same in LIVE.values())} | model call p50 "
          f"{nearest_rank(ok, 50):.0f} ms, p95 {nearest_rank(ok, 95):.0f} ms | outcomes: "
          + ", ".join(f"{k} {n}" for k, n in Counter(g.outcome for a, g, t, same in LIVE.values()).items())
          + f" | replies identical to Day 12's: {sum(same for *_, same in LIVE.values())} of {len(LIVE)}")
    if live.model in PRICES:
        print(f"total: {used_in} input and {used_out} output tokens, about USD "
              f"{(used_in * PRICES[live.model][0] + used_out * PRICES[live.model][1]) / 1e6:.5f} at the prices above")

live pass: 20 questions Day 12's live pass asked (its 6 recorded failures, then 9 answerable and 5 unanswerable ones); gpt-6-luna at most USD 0.0110 (USD 0.1 / 0.5 per million tokens, checked 2026-11-12) | client: attempts 3, timeout 10, deadline 30, backoff_base 0.5, backoff_cap 8, breaker after 3 failed requests for 30 s, fallback: the API-free reader
Live model: off. Fill in RAG_CHAT_* in .env (see .env.example), then set LIVE_MODEL = True.


## 10. Save the run, compare with the accepted baseline

The run holds, for each of the 51 held-out questions, the current pipeline's answer and verdict (as in Day 12's file), what the semantic cache found for it, the cache's curves on dev and test, the fallback's values and both decisions. It is compared with [`evals/baselines/day-13-answers-v5.json`](../evals/baselines/day-13-answers-v5.json): answers, verdicts and cache hits exactly, cosines within 0.005 (another CPU), a hit within 0.001 of θ as a near tie. No timing is stored: timings belong to a machine, not to a baseline.

In [16]:
PIPELINES13 = {"current": {"version": PIPELINE_VERSION, "same as": "names-v2 in evals/baselines/day-12-answers-v4.json",
                           "retriever": "weighted fusion (Day 08, alpha_dense 0.7) of BM25 and dense, each "
                                        "permission-filtered",
                           "context": f"the weighted fusion's top {HITS}", "budget_words": BUDGET,
                           "gate": {"on": "the dense top score", "value": GATE, "from": "Day 07"},
                           "reader": {"code": "rag_from_scratch.queries.NamesReader (Day 10)",
                                      "min_score": READER_THRESHOLD, "names_rule": "Day 10's unknown_names_v2"},
                           "check": "rag_from_scratch.answering.check_citations (Day 06)"}}


def cache_entry(q) -> dict:
    cos, best = nearest_new[q.id]
    hit = best is not None and cos >= THETA
    return {"nearest": best[0] if best else None, "cosine": round(cos, 4), "hit": hit,
            "served_outcome": outcome(best[1], q) if hit else None}


def curve(results: dict) -> dict:
    return {f"{t:.2f}": {"hits": len(r["hits"]), "better": len(r["better"]), "worse": len(r["worse"]),
                         "near": len(r["near"]), "delta": {str(c): d for c, d in r["delta"].items()}}
            for t, r in results.items()}


run_file_data = {
    "name": "Day 13 answer baseline (questions v5)",
    "dataset": {"questions": V5["questions"].relative_to(REPO_ROOT).as_posix(), "version": manifest5["version"],
                "sha256": manifest5["questions_sha256"], "snapshot": manifest5["snapshot"]["fingerprint"]},
    "pipelines": PIPELINES13, "rubric": day12["rubric"],
    "metrics": {"v5 test": {"current": dict(zip(METRICS, counts(GRADES_NEW, NEW)))}},
    "questions": {q.id: {"split": q.split, "type": q.tags[0], "written_from": source_of[q.id],
                         "retrieval": {"top3": list(sent_of(q)), "dense_top": round(rankings["dense"][q.id][0][1], 4),
                                       "first_relevant": first_relevant(rankings["weighted"][q.id], q.answer_chunks)
                                       if q.answerable else None, "margin": round(cutoff_gap(q), 4)},
                         "current": answer_entry(own_new[q.id], GRADES_NEW[q.id],
                                                 closest_call(own_new[q.id], READER, rankings["dense"][q.id][0][1],
                                                              GATE, cutoff_gap(q))),
                         "semantic_cache": cache_entry(q)} for q in NEW},
    "semantic_cache": {"theta": THETA, "key": "the dense query vector (bge-small, query instruction)",
                       "scope": "the asker's groups and the index fingerprint", "entries": "answered answers of the "
                       "236 dev questions", "dev (in-sample, leave one out)": curve(dev_curve),
                       "v5 test": {k: D1_RESULT[k] for k in ("hits", "better", "worse", "near")}
                       | {"delta": {str(c): d for c, d in D1_RESULT["delta"].items()}},
                       "v5 test, every threshold (description)": curve(test_curve)},
    "fallback": {"dev (in-sample)": dict(zip(METRICS[:5], dev_counts[:5])) | {"value": {str(c): v for c, v in
                                                                                      dev_values.items()}},
                 "v5 test": {"value": {str(c): v for c, v in D2_VALUE.items()}}},
    "decisions": {"preregistration": PREREGISTRATION.relative_to(REPO_ROOT).as_posix(),
                  "D1 a semantic answer cache (theta 0.89)": "adopted" if D1 else "not adopted",
                  "D2 the API-free reader as the fallback when the model fails": "adopted" if D2 else "not adopted"},
}


def compare_runs(new: dict, old: dict) -> tuple[list[str], list[str]]:
    # Day 12's compare_answers() for the pipeline's answers, verdicts, metrics and decisions; then the cache: every hit
    # or miss and nearest question exactly, cosines within 0.005 (another CPU), a flipped hit within TIE of theta a tie
    changes, ties = compare_answers(new, old, tuple(new["pipelines"]))
    for qid, now in new["questions"].items():
        then = old["questions"].get(qid, {}).get("semantic_cache")
        if then is None:
            continue
        c_now, c_then = now["semantic_cache"], then
        if abs(c_now["cosine"] - c_then["cosine"]) > 0.005 or c_now["nearest"] != c_then["nearest"]:
            changes.append(f"{qid}: nearest dev question {c_then['nearest']} {c_then['cosine']} -> "
                           f"{c_now['nearest']} {c_now['cosine']}")
        elif (c_now["hit"], c_now["served_outcome"]) != (c_then["hit"], c_then["served_outcome"]):
            (ties if min(abs(c["cosine"] - new["semantic_cache"]["theta"]) for c in (c_now, c_then)) < TIE
             else changes).append(f"{qid}: cache {c_then['hit']} -> {c_now['hit']}")
    for part in ("semantic_cache", "fallback"):
        if new[part] != old[part]:
            (ties if ties and not changes else changes).append(f"{part} changed")
    return changes, ties


OUT_DIR.mkdir(parents=True, exist_ok=True)
run_file = OUT_DIR / BASELINE_FILE.name
run_file.write_text(json.dumps(run_file_data, indent=1, ensure_ascii=False) + "\n", encoding="utf-8", newline="\n")
print(f"this run: {run_file.relative_to(REPO_ROOT).as_posix()} ({len(NEW)} test questions: answers, verdicts, the "
      "semantic cache's lookups; the cache's dev curve; both decisions)")
if BASELINE_FILE.is_file():
    changes, ties = compare_runs(run_file_data, json.loads(BASELINE_FILE.read_text(encoding="utf-8")))
    print(f"compared with {BASELINE_FILE.relative_to(REPO_ROOT).as_posix()}: "
          + (f"{len(changes)} changes" + "".join(f"\n  {c}" for c in changes) if changes else
             f"no changes ({len(NEW)} questions: status, answer text, citations, verdict, cache lookups; metrics, the "
             "cache's curves, the fallback's values and the decisions)"))
    for t in ties:
        print(f"  near tie, not a change: {t}")
else:
    changes, ties = None, []
    print(f"no accepted baseline yet: review {run_file.name}, then copy it to "
          f"{BASELINE_FILE.relative_to(REPO_ROOT).as_posix()} and commit it")
closest = sorted((run_file_data["questions"][q.id]["current"]["closest_call"][0] or math.inf, q.id) for q in NEW)
print("closest calls: " + ", ".join(f"{qid} {gap:.4f}" for gap, qid in closest[:3]) + " | closest cache decisions: "
      + ", ".join(f"{q.id} {abs(nearest_new[q.id][0] - THETA):.4f}" for q in
                  sorted(NEW, key=lambda q: abs(nearest_new[q.id][0] - THETA))[:3]))

this run: outputs/day-13/day-13-answers-v5.json (51 test questions: answers, verdicts, the semantic cache's lookups; the cache's dev curve; both decisions)
compared with evals/baselines/day-13-answers-v5.json: no changes (51 questions: status, answer text, citations, verdict, cache lookups; metrics, the cache's curves, the fallback's values and the decisions)
closest calls: d13-a08 0.0001, d13-a05 0.0002, d13-a06 0.0002 | closest cache decisions: d13-a07 0.0033, d13-a13 0.0035, d13-u16 0.0070


## 11. Sanity checks

Every number the lesson quotes, every decision, and every behaviour shown above, checked again. The timings are checked only for their shape (a cold reader slower than a warm one), never for their value.

In [17]:
def ids(qs) -> list[str]:
    return [q.id for q in qs]


# Section 1: the same data, and Day 12's accepted answers reproduced exactly (both pipelines)
assert len(queries5) == 317 and len(NEW) == 51 and len(DEV) == 236 and same266 and not problems
assert sum(q.answerable for q in NEW) == 33 and Counter(q.tags[0] for q in NEW) == {"paraphrase": 30, "near-neighbour": 21}
assert changes12 == [] and ties12 == [] and same_retrieval
# Section 2: tracing changes nothing and leaks nothing; the shown trace diagnoses the lost laptop without text
assert len(TRACES) == len(natural) and not trace_leaks
assert all(t["question_sha256"] == sha16(q.question) and q.question not in json.dumps(t) for q, t in zip(natural, TRACES))
assert trace["cited"] == ["travel-policy#safety-while-travelling"] and \
    "security-incident-reporting#how-to-report" in next(s["sent"] for s in trace["spans"] if s["name"] == "context")
assert {s["name"] for t in TRACES for s in t["spans"]} == set(STEPS) - {"cache.answers"}
# Section 3: the cold request embeds its question twice; precomputed sentence vectors remove the reader's embedding
assert all(t["ms"] >= s["ms"] for t in cold + warm + repeat for s in t["spans"])
assert LATENCY["cold"]["steps"]["generate"][1] > 10 * LATENCY["warm"]["steps"]["generate"][1]
assert LATENCY["repeat"]["p95"] < LATENCY["warm"]["p50"] and len(repeat) == len(DEV)
assert len(seconds) == 40 and sorted(seconds)[19] == nearest_rank(seconds, 50)
# Section 4: the recorded tokens, and what the estimate gets right
assert (spent_in, spent_out) == (18075, 1133) and 0.8 < statistics.mean(ratios) < 1.0
assert max(e for _, _, e in rows) < PROMPT_CACHE_MIN_TOKENS and len(gated) == 29 and reach == 207
# Section 5: exact caches change nothing; the two key mistakes do what they show
assert exact.hits == len(natural) and LEAKED == ["manager-pay-review-guide#overview"]
assert SAFE == ("abstained", [], "gate", True, "answer cache") and RECHECK is None
assert STALE["NoFingerprintCache"][1] == STALE["NoFingerprintCache"][0] and "14 characters" in STALE["NoFingerprintCache"][1]
assert "16 characters" in STALE["AnswerCache"][1] and STALE["AnswerCache"][2] != "answer cache"
# D1: the rule's value, and the decision as pre-registered (hits >= 1 and none worse after near ties)
assert THETA == 0.89 and len(dev_curve[THETA]["hits"]) == 0 and not dev_curve[THETA]["worse"]
assert D1 == (len(D1_RESULT["hits"]) >= 1 and not D1_RESULT["worse"])
assert all(test_curve[t]["hits"] == D1_RESULT["hits"] for t in (THETA,))
assert all(len(test_curve[a]["hits"]) >= len(test_curve[b]["hits"]) for a, b in zip(THETAS, THETAS[1:]))
assert all(nearest_new[q.id][1] is None or by_qid[nearest_new[q.id][1][0]].user == q.user or
           DIRECTORY[by_qid[nearest_new[q.id][1][0]].user] == DIRECTORY[q.user] for q in NEW)
# Section 6: every failure handled as designed (attempt counts don't depend on timing, except the slow server's)
expected = {"healthy": ["ok"], "one 503, then fine": ["HTTP 503", "ok"], "429 with Retry-After 0.2 s": ["HTTP 429", "ok"],
            "429 with Retry-After 5 s": ["HTTP 429"], "429, no credit left": ["HTTP 429"], "401, wrong key": ["HTTP 401"],
            "500 three times": ["HTTP 500"] * 3, "trickle (2.5 s per reply)": ["deadline"],
            "not JSON (a proxy page)": ["not JSON"]}
assert all(RESILIENCE[name]["attempts"] == attempts for name, attempts in expected.items())
assert set(RESILIENCE["slow (2 s per reply)"]["attempts"]) == {"timeout"}
assert RESILIENCE["429 with Retry-After 0.2 s"]["waited"] == 0.2
jitter = [(s["reason"], s["wait_ms"]) for s in RESILIENCE["500 three times"]["trace"]["spans"] if s["name"] == "backoff"]
assert [r for r, _ in jitter] == ["jitter", "jitter"] and \
    all(0 <= ms < 1000 * min(DEMO["backoff_cap"], DEMO["backoff_base"] * 2 ** n) for n, (_, ms) in enumerate(jitter))
assert all(RESILIENCE[n]["served_by"] == "gpt-6-luna via openai" for n in ("healthy", "one 503, then fine",
                                                                           "429 with Retry-After 0.2 s"))
assert all("fallback after" in r["served_by"] and r["status"] in ("answered", "abstained", "blocked")
           for n, r in RESILIENCE.items() if n not in ("healthy", "one 503, then fine", "429 with Retry-After 0.2 s"))
assert RESILIENCE["trickle (2.5 s per reply)"]["took"] < DEMO["deadline"] + DEMO["timeout"] < DAY06_TRICKLE
assert "authentication failed" in NO_FALLBACK and FAKE_KEY not in NO_FALLBACK
assert all(FAKE_KEY not in json.dumps(r["trace"]) and not leaks(r["trace"], q_demo.question, [FAKE_KEY])
           for r in RESILIENCE.values())
assert OUTAGE["retries only"][0] == 24 and OUTAGE["retries and a breaker"][0] == 9 and \
    BREAKER_AFTER == ("open", 1, "gpt-6-luna via openai", "closed")
# D2: the dev values as pre-registered, and the decision
assert D2 == (D2_VALUE[COST] > 0) and dev_values == PREREGISTERED["dev value"]
# Section 8: the new regression cases behave as pinned, and are in the registry
assert NEW_CASES["d12-a05"] == ("blocked", ["12 is not in the cited source"])
assert NEW_CASES["d12-a20"] == ("blocked", ["no citation"])
assert NEW_CASES["d03-13"] == ("answered", "wrong", True, False)
assert {"citation_check_false_block", "model_ignores_answer", "timeout_is_not_a_deadline"} <= {r["check"] for r in regressions}
# Section 9: the live pass is off unless switched on, and never in the committed notebook
assert LIVE_MODEL or LIVE == {}
# Section 10: the accepted run
assert changes in (None, []) and run_file_data["decisions"]["D1 a semantic answer cache (theta 0.89)"] == \
    ("adopted" if D1 else "not adopted")
print("All Day 13 sanity checks passed.")

All Day 13 sanity checks passed.


## 12. Exercises

Each takes 5–15 minutes. Restart and rerun the notebook after changing anything. Today's test split has been used to decide, so treat everything here as dev.

1. **Find the duplicate work.** In section 2, trace one request with the cold reader and count how many times the question is embedded. Then pass `queries=QueryVectorCache(embedder)` and the cold reader built on it: what changes in the trace?
2. **A tighter deadline.** Set `DEMO["deadline"] = 0.6` and rerun section 6. Which scenarios change from the model to the fallback, and why does "429 with Retry-After 0.2 s" survive?
3. **Retry a billing error.** Add 429 with `credit_balance_exhausted` to the retried codes. How many calls does the "no credit left" scenario make now, and what would that do to a real bill or rate limit?
4. **Break the semantic cache's scope.** Make `SemanticCache.nearest()` ignore groups. Which held-out question could be served a managers-only answer, and does the permission re-check in `get()` still stop it?
5. **A cheaper prompt.** Shorten `INSTRUCTIONS` (a copy, not `chat.py`) by a third and recompute section 4's estimate. What is saved per 1,000 requests, and which of Day 06's six rules would you be least willing to drop?

LEARN.md's exercise asks you to add a trace check of your own.